# Financial Audit & Risk Analytics Platform

## End-to-End Data Analyst Project for Audit, Risk Advisory and Financial Control

---

**Author:** Chinez Benidir  
**Target roles:** Data Analyst, Audit Analytics Analyst, Risk Advisory Analyst, Financial Data Analyst  
**Business Context:** Financial audit, internal control, payment monitoring and supplier risk analysis  
**Objective:** Detect financial anomalies, identify risky suppliers, automate audit controls, build risk scores and generate business recommendations.

---

## Project Overview

This project simulates a real audit analytics workflow used by consulting and audit firms such as KPMG, EY and PwC.

The objective is to analyze financial transactions, invoices, suppliers and payments in order to identify:

- duplicate invoices;
- suspicious payments;
- high-risk suppliers;
- abnormal transaction amounts;
- late payments;
- payments without purchase orders;
- policy breaches;
- concentration of expenses by supplier or department;
- internal control weaknesses.

The project includes data quality checks, audit rule automation, risk scoring, anomaly detection, SQL data mart creation, automated Excel reporting and an interactive dashboard.

---

## Table of Contents

1. Project Setup and Libraries  
2. Financial Data Creation and Loading  
3. Data Quality Assessment  
4. Data Cleaning and Standardization  
5. Audit Feature Engineering  
6. Audit Rules Engine  
7. Transaction Risk Scoring  
8. Supplier Risk Scoring  
9. Financial KPI Analysis  
10. Department and Supplier Risk Analysis  
11. Machine Learning Anomaly Detection  
12. SQL Data Mart Creation  
13. Automated Excel Audit Report  
14. Interactive Audit Risk Dashboard  
15. Executive Summary and Business Recommendations  

In [1]:
# ============================================================
# 1. PROJECT SETUP AND LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path
from datetime import datetime, timedelta
import random
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Display settings
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
DASHBOARD_DIR = PROJECT_ROOT / "dashboard"
SQL_DIR = PROJECT_ROOT / "sql"
SCREENSHOTS_DIR = PROJECT_ROOT / "screenshots"

directories = [
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    REPORTS_DIR,
    FIGURES_DIR,
    DASHBOARD_DIR,
    SQL_DIR,
    SCREENSHOTS_DIR
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully")
print("Project directories verified successfully")
print("-" * 60)
print("Project root       :", PROJECT_ROOT)
print("Raw data           :", RAW_DATA_DIR)
print("Processed data     :", PROCESSED_DATA_DIR)
print("Reports            :", REPORTS_DIR)
print("Figures            :", FIGURES_DIR)
print("SQL                :", SQL_DIR)
print("Dashboard          :", DASHBOARD_DIR)
print("Screenshots        :", SCREENSHOTS_DIR)

Libraries imported successfully
Project directories verified successfully
------------------------------------------------------------
Project root       : ..
Raw data           : ..\data\raw
Processed data     : ..\data\processed
Reports            : ..\reports
Figures            : ..\reports\figures
SQL                : ..\sql
Dashboard          : ..\dashboard
Screenshots        : ..\screenshots


# 2. Financial Data Creation and Loading

This section creates a realistic synthetic financial audit dataset.

The objective is to simulate the type of data used in audit analytics, internal control review and risk advisory assignments.

The dataset includes:

- suppliers;
- departments;
- purchase orders;
- supplier invoices;
- payments.

Controlled anomalies are intentionally introduced in the data to support audit rule testing and risk scoring, including duplicate invoices, missing purchase orders, late payments, payment timing issues, round amounts and high-value transactions.

In [2]:
# ============================================================
# 2. FINANCIAL DATA CREATION AND LOADING
# ============================================================

print("=" * 80)
print("FINANCIAL DATA CREATION AND LOADING")
print("=" * 80)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

rng = np.random.default_rng(42)
random.seed(42)

# ------------------------------------------------------------
# Dataset size
# ------------------------------------------------------------

n_suppliers = 320
n_departments = 12
n_purchase_orders = 5200
n_invoices = 8500

start_date = pd.Timestamp("2024-01-01")
end_date = pd.Timestamp("2024-12-31")

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def random_dates(start, end, size):
    start_u = start.value // 10**9
    end_u = end.value // 10**9
    return pd.to_datetime(rng.integers(start_u, end_u, size), unit="s").normalize()

def generate_id(prefix, number, width=6):
    return f"{prefix}{str(number).zfill(width)}"

# ------------------------------------------------------------
# Departments dimension
# ------------------------------------------------------------

department_names = [
    "Finance",
    "Procurement",
    "IT",
    "Operations",
    "Sales",
    "Marketing",
    "Human Resources",
    "Legal",
    "Risk Management",
    "Customer Support",
    "Logistics",
    "General Administration"
]

business_units = [
    "Corporate",
    "Commercial",
    "Technology",
    "Operations",
    "Support"
]

departments_df = pd.DataFrame({
    "department_id": [generate_id("DPT", i + 1, 3) for i in range(n_departments)],
    "department_name": department_names,
    "business_unit": rng.choice(business_units, n_departments),
    "approval_threshold": rng.choice(
        [5000, 10000, 25000, 50000, 100000],
        n_departments,
        p=[0.20, 0.30, 0.25, 0.15, 0.10]
    )
})

# ------------------------------------------------------------
# Suppliers dimension
# ------------------------------------------------------------

supplier_categories = [
    "IT Services",
    "Consulting",
    "Office Supplies",
    "Logistics",
    "Marketing Services",
    "Maintenance",
    "Professional Services",
    "Telecom",
    "Training",
    "Facilities"
]

supplier_countries = [
    "Algeria",
    "France",
    "Spain",
    "Germany",
    "United Kingdom",
    "United Arab Emirates",
    "Turkey",
    "Tunisia",
    "Morocco"
]

suppliers_df = pd.DataFrame({
    "supplier_id": [generate_id("SUP", i + 1, 5) for i in range(n_suppliers)],
    "supplier_name": [f"Supplier_{str(i + 1).zfill(4)}" for i in range(n_suppliers)],
    "supplier_category": rng.choice(supplier_categories, n_suppliers),
    "supplier_country": rng.choice(
        supplier_countries,
        n_suppliers,
        p=[0.58, 0.10, 0.06, 0.05, 0.04, 0.05, 0.05, 0.04, 0.03]
    ),
    "onboarding_date": random_dates(pd.Timestamp("2015-01-01"), pd.Timestamp("2024-09-30"), n_suppliers),
    "payment_terms_days": rng.choice([15, 30, 45, 60, 90], n_suppliers, p=[0.10, 0.45, 0.25, 0.15, 0.05]),
    "tax_id": [f"TAX-{rng.integers(10000000, 99999999)}" for _ in range(n_suppliers)],
    "bank_account": [f"DZ{rng.integers(100000000000, 999999999999)}" for _ in range(n_suppliers)]
})

# Introduce missing supplier tax IDs for audit data quality checks
missing_tax_suppliers = rng.choice(suppliers_df.index, size=18, replace=False)
suppliers_df.loc[missing_tax_suppliers, "tax_id"] = np.nan

# ------------------------------------------------------------
# Supplier selection probabilities
# Some suppliers are intentionally more frequent to simulate supplier concentration
# ------------------------------------------------------------

supplier_weights = np.ones(n_suppliers)
supplier_weights[:10] = 18
supplier_weights[10:30] = 6
supplier_weights = supplier_weights / supplier_weights.sum()

department_weights = np.array([0.13, 0.14, 0.12, 0.12, 0.08, 0.07, 0.06, 0.05, 0.08, 0.05, 0.06, 0.04])
department_weights = department_weights / department_weights.sum()

# ------------------------------------------------------------
# Purchase orders fact table
# ------------------------------------------------------------

po_supplier_ids = rng.choice(suppliers_df["supplier_id"], n_purchase_orders, p=supplier_weights)
po_department_ids = rng.choice(departments_df["department_id"], n_purchase_orders, p=department_weights)
po_dates = random_dates(start_date, end_date - pd.Timedelta(days=30), n_purchase_orders)

# Lognormal distribution gives realistic skewed financial amounts
po_amounts = rng.lognormal(mean=9.0, sigma=1.0, size=n_purchase_orders)
po_amounts = np.clip(po_amounts, 200, 450000).round(2)

purchase_orders_df = pd.DataFrame({
    "po_id": [generate_id("PO", i + 1, 7) for i in range(n_purchase_orders)],
    "supplier_id": po_supplier_ids,
    "department_id": po_department_ids,
    "po_date": po_dates,
    "po_amount": po_amounts,
    "approval_status": rng.choice(["Approved", "Pending", "Rejected"], n_purchase_orders, p=[0.88, 0.08, 0.04]),
    "buyer": rng.choice(
        ["Buyer_A", "Buyer_B", "Buyer_C", "Buyer_D", "Buyer_E", "Buyer_F"],
        n_purchase_orders
    )
})

# ------------------------------------------------------------
# Invoices fact table
# ------------------------------------------------------------

invoice_supplier_ids = rng.choice(suppliers_df["supplier_id"], n_invoices, p=supplier_weights)
invoice_department_ids = rng.choice(departments_df["department_id"], n_invoices, p=department_weights)

# 82% of invoices linked to a purchase order
po_link_flag = rng.choice([1, 0], n_invoices, p=[0.82, 0.18])

linked_po_ids = rng.choice(purchase_orders_df["po_id"], n_invoices)
invoice_po_ids = np.where(po_link_flag == 1, linked_po_ids, None)

invoice_dates = random_dates(start_date, end_date, n_invoices)

# Base invoice amounts
invoice_amounts = rng.lognormal(mean=8.7, sigma=1.15, size=n_invoices)
invoice_amounts = np.clip(invoice_amounts, 100, 600000).round(2)

# Some invoices are intentionally round amounts
round_amount_indices = rng.choice(np.arange(n_invoices), size=420, replace=False)
invoice_amounts[round_amount_indices] = rng.choice(
    [5000, 10000, 15000, 20000, 25000, 50000, 100000],
    size=len(round_amount_indices)
)

# Retrieve supplier payment terms
supplier_terms_map = suppliers_df.set_index("supplier_id")["payment_terms_days"].to_dict()
invoice_payment_terms = [supplier_terms_map[supplier_id] for supplier_id in invoice_supplier_ids]

due_dates = invoice_dates + pd.to_timedelta(invoice_payment_terms, unit="D")

invoices_df = pd.DataFrame({
    "invoice_id": [generate_id("INV", i + 1, 7) for i in range(n_invoices)],
    "invoice_number": [f"FCT-{rng.integers(100000, 999999)}" for _ in range(n_invoices)],
    "supplier_id": invoice_supplier_ids,
    "department_id": invoice_department_ids,
    "po_id": invoice_po_ids,
    "invoice_date": invoice_dates,
    "invoice_amount": invoice_amounts,
    "currency": rng.choice(["DZD", "EUR", "USD"], n_invoices, p=[0.82, 0.12, 0.06]),
    "payment_terms_days": invoice_payment_terms,
    "due_date": due_dates,
    "invoice_status": rng.choice(["Validated", "Pending Review", "Disputed"], n_invoices, p=[0.86, 0.10, 0.04])
})

# ------------------------------------------------------------
# Controlled duplicate invoices
# Duplicate logic: same supplier, same invoice number, same or very close amount
# ------------------------------------------------------------

duplicate_source_indices = rng.choice(invoices_df.index, size=130, replace=False)

duplicate_invoices = invoices_df.loc[duplicate_source_indices].copy()
duplicate_invoices["invoice_id"] = [
    generate_id("INV", n_invoices + i + 1, 7) for i in range(len(duplicate_invoices))
]

# Slight date shift on some duplicates
duplicate_invoices["invoice_date"] = duplicate_invoices["invoice_date"] + pd.to_timedelta(
    rng.choice([0, 1, 2, 3], size=len(duplicate_invoices)),
    unit="D"
)

duplicate_invoices["due_date"] = duplicate_invoices["invoice_date"] + pd.to_timedelta(
    duplicate_invoices["payment_terms_days"],
    unit="D"
)

duplicate_invoices["invoice_status"] = rng.choice(
    ["Validated", "Pending Review"],
    len(duplicate_invoices),
    p=[0.80, 0.20]
)

invoices_df = pd.concat([invoices_df, duplicate_invoices], ignore_index=True)

# ------------------------------------------------------------
# Payments fact table
# ------------------------------------------------------------

paid_flag = rng.choice([1, 0], len(invoices_df), p=[0.92, 0.08])

paid_invoices = invoices_df[paid_flag == 1].copy()

# Payment delays: most around due date, some late, some early
payment_delay_days = rng.normal(loc=5, scale=18, size=len(paid_invoices)).round().astype(int)

# Force some late payments
late_indices = rng.choice(np.arange(len(paid_invoices)), size=650, replace=False)
payment_delay_days[late_indices] = rng.integers(35, 120, size=len(late_indices))

# Force some payments before invoice date
early_indices = rng.choice(np.arange(len(paid_invoices)), size=95, replace=False)
payment_dates = paid_invoices["due_date"].reset_index(drop=True) + pd.to_timedelta(payment_delay_days, unit="D")
payment_dates.iloc[early_indices] = (
    paid_invoices["invoice_date"].reset_index(drop=True).iloc[early_indices]
    - pd.to_timedelta(rng.integers(1, 10, size=len(early_indices)), unit="D")
)

payment_amounts = paid_invoices["invoice_amount"].reset_index(drop=True).copy()

# Small payment differences
payment_amounts = payment_amounts * rng.normal(loc=1.0, scale=0.015, size=len(payment_amounts))
payment_amounts = payment_amounts.clip(lower=0).round(2)

payments_df = pd.DataFrame({
    "payment_id": [generate_id("PAY", i + 1, 7) for i in range(len(paid_invoices))],
    "invoice_id": paid_invoices["invoice_id"].values,
    "supplier_id": paid_invoices["supplier_id"].values,
    "department_id": paid_invoices["department_id"].values,
    "payment_date": payment_dates.values,
    "payment_amount": payment_amounts.values,
    "payment_method": rng.choice(
        ["Bank Transfer", "Check", "Card", "Cash"],
        len(paid_invoices),
        p=[0.86, 0.08, 0.04, 0.02]
    ),
    "payment_status": rng.choice(
        ["Completed", "Pending", "Failed"],
        len(paid_invoices),
        p=[0.94, 0.04, 0.02]
    )
})

# Introduce weekend payments by shifting some payment dates
weekend_payment_indices = rng.choice(payments_df.index, size=260, replace=False)
payments_df.loc[weekend_payment_indices, "payment_date"] = payments_df.loc[
    weekend_payment_indices,
    "payment_date"
] + pd.to_timedelta(
    5 - pd.to_datetime(payments_df.loc[weekend_payment_indices, "payment_date"]).dt.dayofweek,
    unit="D"
)

# ------------------------------------------------------------
# Save raw datasets
# ------------------------------------------------------------

suppliers_path = RAW_DATA_DIR / "suppliers.csv"
departments_path = RAW_DATA_DIR / "departments.csv"
purchase_orders_path = RAW_DATA_DIR / "purchase_orders.csv"
invoices_path = RAW_DATA_DIR / "invoices.csv"
payments_path = RAW_DATA_DIR / "payments.csv"

suppliers_df.to_csv(suppliers_path, index=False)
departments_df.to_csv(departments_path, index=False)
purchase_orders_df.to_csv(purchase_orders_path, index=False)
invoices_df.to_csv(invoices_path, index=False)
payments_df.to_csv(payments_path, index=False)

# ------------------------------------------------------------
# Display outputs
# ------------------------------------------------------------

print("Financial audit datasets created successfully")
print("-" * 80)
print("Suppliers shape       :", suppliers_df.shape)
print("Departments shape     :", departments_df.shape)
print("Purchase orders shape :", purchase_orders_df.shape)
print("Invoices shape        :", invoices_df.shape)
print("Payments shape        :", payments_df.shape)

print("\nFiles saved successfully:")
print("-", suppliers_path)
print("-", departments_path)
print("-", purchase_orders_path)
print("-", invoices_path)
print("-", payments_path)

print("\nSuppliers preview:")
display(suppliers_df.head())

print("\nDepartments preview:")
display(departments_df.head())

print("\nInvoices preview:")
display(invoices_df.head())

print("\nPayments preview:")
display(payments_df.head())

FINANCIAL DATA CREATION AND LOADING
Financial audit datasets created successfully
--------------------------------------------------------------------------------
Suppliers shape       : (320, 8)
Departments shape     : (12, 4)
Purchase orders shape : (5200, 7)
Invoices shape        : (8630, 11)
Payments shape        : (7918, 8)

Files saved successfully:
- ..\data\raw\suppliers.csv
- ..\data\raw\departments.csv
- ..\data\raw\purchase_orders.csv
- ..\data\raw\invoices.csv
- ..\data\raw\payments.csv

Suppliers preview:


,supplier_id,supplier_name,supplier_category,supplier_country,onboarding_date,payment_terms_days,tax_id,bank_account
0,SUP00001,Supplier_0001,Training,Turkey,2020-02-17,45,TAX-14868708,DZ492684993724
1,SUP00002,Supplier_0002,Training,Germany,2018-07-07,90,TAX-77575471,DZ797161493162
2,SUP00003,Supplier_0003,Office Supplies,Turkey,2024-05-16,45,TAX-67358568,DZ131354515251
3,SUP00004,Supplier_0004,Professional Services,Turkey,2019-12-31,45,TAX-81427183,DZ687031845818
4,SUP00005,Supplier_0005,Consulting,Algeria,2022-11-17,30,TAX-84459703,DZ840814585673



Departments preview:


,department_id,department_name,business_unit,approval_threshold
0,DPT001,Finance,Corporate,50000
1,DPT002,Procurement,Operations,50000
2,DPT003,IT,Operations,5000
3,DPT004,Operations,Technology,10000
4,DPT005,Sales,Technology,10000



Invoices preview:


,invoice_id,invoice_number,supplier_id,department_id,po_id,invoice_date,invoice_amount,currency,payment_terms_days,due_date,invoice_status
0,INV0000001,FCT-634138,SUP00077,DPT009,PO0004332,2024-03-06,"1,082.2000",EUR,30,2024-04-05,Validated
1,INV0000002,FCT-562234,SUP00010,DPT004,PO0002739,2024-07-06,"17,142.8700",DZD,15,2024-07-21,Validated
2,INV0000003,FCT-647518,SUP00011,DPT002,None,2024-06-22,"1,411.5100",DZD,30,2024-07-22,Validated
3,INV0000004,FCT-802936,SUP00017,DPT006,None,2024-10-24,"3,925.9000",DZD,90,2025-01-22,Validated
4,INV0000005,FCT-449867,SUP00079,DPT008,PO0001366,2024-09-19,"15,000.0000",DZD,60,2024-11-18,Pending Review



Payments preview:


,payment_id,invoice_id,supplier_id,department_id,payment_date,payment_amount,payment_method,payment_status
0,PAY0000001,INV0000001,SUP00077,DPT009,2024-06-11,"1,071.9700",Bank Transfer,Completed
1,PAY0000002,INV0000002,SUP00010,DPT004,2024-07-31,"17,212.1800",Card,Completed
2,PAY0000003,INV0000003,SUP00011,DPT002,2024-07-19,"1,372.6200",Bank Transfer,Completed
3,PAY0000004,INV0000004,SUP00017,DPT006,2025-02-10,"3,979.9800",Bank Transfer,Completed
4,PAY0000005,INV0000005,SUP00079,DPT008,2024-12-20,"15,345.5000",Bank Transfer,Completed


# 3. Data Quality Assessment

This section performs a data quality and audit readiness assessment.

The objective is to evaluate whether the financial datasets are complete, consistent and reliable enough for audit analytics.

The assessment covers:

- table dimensions;
- missing values;
- duplicate records;
- data type validation;
- referential integrity checks;
- invoice and payment consistency checks;
- missing purchase orders;
- supplier master data quality;
- payment timing issues;
- high-level audit risk indicators.

In [9]:
# ============================================================
# 3. DATA QUALITY ASSESSMENT
# ============================================================

print("=" * 80)
print("DATA QUALITY ASSESSMENT")
print("=" * 80)

# ------------------------------------------------------------
# Ensure date columns are datetime
# ------------------------------------------------------------

suppliers_df["onboarding_date"] = pd.to_datetime(suppliers_df["onboarding_date"])

purchase_orders_df["po_date"] = pd.to_datetime(purchase_orders_df["po_date"])

invoices_df["invoice_date"] = pd.to_datetime(invoices_df["invoice_date"])
invoices_df["due_date"] = pd.to_datetime(invoices_df["due_date"])

payments_df["payment_date"] = pd.to_datetime(payments_df["payment_date"])

# ------------------------------------------------------------
# Table overview
# ------------------------------------------------------------

tables = {
    "suppliers": suppliers_df,
    "departments": departments_df,
    "purchase_orders": purchase_orders_df,
    "invoices": invoices_df,
    "payments": payments_df
}

table_overview = []

for table_name, df in tables.items():
    table_overview.append({
        "Table": table_name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Duplicate_Rows": df.duplicated().sum(),
        "Missing_Values_Total": df.isna().sum().sum()
    })

table_overview_df = pd.DataFrame(table_overview)

print("Table overview:")
display(table_overview_df)

# ------------------------------------------------------------
# Missing values report
# ------------------------------------------------------------

missing_reports = []

for table_name, df in tables.items():
    missing_count = df.isna().sum()
    missing_percentage = (missing_count / len(df) * 100).round(2)
    
    temp_missing_report = pd.DataFrame({
        "Table": table_name,
        "Column": missing_count.index,
        "Missing_Count": missing_count.values,
        "Missing_Percentage": missing_percentage.values
    })
    
    missing_reports.append(temp_missing_report)

missing_values_report = pd.concat(missing_reports, ignore_index=True)
missing_values_report = missing_values_report.sort_values(
    ["Missing_Count", "Missing_Percentage"],
    ascending=False
)

print("\nMissing values report:")
display(missing_values_report[missing_values_report["Missing_Count"] > 0])

# ------------------------------------------------------------
# Data types report
# ------------------------------------------------------------

data_types_report = []

for table_name, df in tables.items():
    for column in df.columns:
        data_types_report.append({
            "Table": table_name,
            "Column": column,
            "Data_Type": str(df[column].dtype),
            "Unique_Values": df[column].nunique(dropna=True)
        })

data_types_report = pd.DataFrame(data_types_report)

print("\nData types report sample:")
display(data_types_report.head(30))

# ------------------------------------------------------------
# Date range checks
# ------------------------------------------------------------

date_range_report = pd.DataFrame({
    "Table": [
        "suppliers",
        "purchase_orders",
        "invoices",
        "invoices",
        "payments"
    ],
    "Date_Column": [
        "onboarding_date",
        "po_date",
        "invoice_date",
        "due_date",
        "payment_date"
    ],
    "Min_Date": [
        suppliers_df["onboarding_date"].min(),
        purchase_orders_df["po_date"].min(),
        invoices_df["invoice_date"].min(),
        invoices_df["due_date"].min(),
        payments_df["payment_date"].min()
    ],
    "Max_Date": [
        suppliers_df["onboarding_date"].max(),
        purchase_orders_df["po_date"].max(),
        invoices_df["invoice_date"].max(),
        invoices_df["due_date"].max(),
        payments_df["payment_date"].max()
    ]
})

print("\nDate range report:")
display(date_range_report)

# ------------------------------------------------------------
# Referential integrity checks
# ------------------------------------------------------------

valid_supplier_ids = set(suppliers_df["supplier_id"])
valid_department_ids = set(departments_df["department_id"])
valid_po_ids = set(purchase_orders_df["po_id"])
valid_invoice_ids = set(invoices_df["invoice_id"])

invalid_invoice_supplier = ~invoices_df["supplier_id"].isin(valid_supplier_ids)
invalid_invoice_department = ~invoices_df["department_id"].isin(valid_department_ids)

invoice_po_not_null = invoices_df["po_id"].notna()
invalid_invoice_po = invoice_po_not_null & ~invoices_df["po_id"].isin(valid_po_ids)

invalid_payment_invoice = ~payments_df["invoice_id"].isin(valid_invoice_ids)
invalid_payment_supplier = ~payments_df["supplier_id"].isin(valid_supplier_ids)
invalid_payment_department = ~payments_df["department_id"].isin(valid_department_ids)

referential_integrity_report = pd.DataFrame({
    "Check": [
        "Invoices with invalid supplier_id",
        "Invoices with invalid department_id",
        "Invoices with invalid po_id",
        "Payments with invalid invoice_id",
        "Payments with invalid supplier_id",
        "Payments with invalid department_id"
    ],
    "Number_of_Records": [
        invalid_invoice_supplier.sum(),
        invalid_invoice_department.sum(),
        invalid_invoice_po.sum(),
        invalid_payment_invoice.sum(),
        invalid_payment_supplier.sum(),
        invalid_payment_department.sum()
    ]
})

referential_integrity_report["Percentage"] = (
    referential_integrity_report["Number_of_Records"] / 
    [len(invoices_df), len(invoices_df), len(invoices_df), len(payments_df), len(payments_df), len(payments_df)] 
    * 100
).round(2)

print("\nReferential integrity report:")
display(referential_integrity_report)

# ------------------------------------------------------------
# Invoice audit readiness checks
# ------------------------------------------------------------

invoice_without_po = invoices_df["po_id"].isna()
invoice_zero_or_negative_amount = invoices_df["invoice_amount"] <= 0
invoice_due_before_invoice = invoices_df["due_date"] < invoices_df["invoice_date"]
invoice_round_amount = invoices_df["invoice_amount"] % 1000 == 0
invoice_high_value = invoices_df["invoice_amount"] >= 100000

# Potential duplicate invoices:
# same supplier, same invoice number, same invoice amount
duplicate_invoice_key = invoices_df.duplicated(
    subset=["supplier_id", "invoice_number", "invoice_amount"],
    keep=False
)

invoice_audit_checks = pd.DataFrame({
    "Audit_Check": [
        "Invoices without purchase order",
        "Zero or negative invoice amount",
        "Due date before invoice date",
        "Round amount invoices",
        "High-value invoices >= 100,000",
        "Potential duplicate invoices"
    ],
    "Number_of_Records": [
        invoice_without_po.sum(),
        invoice_zero_or_negative_amount.sum(),
        invoice_due_before_invoice.sum(),
        invoice_round_amount.sum(),
        invoice_high_value.sum(),
        duplicate_invoice_key.sum()
    ]
})

invoice_audit_checks["Percentage"] = (
    invoice_audit_checks["Number_of_Records"] / len(invoices_df) * 100
).round(2)

print("\nInvoice audit checks:")
display(invoice_audit_checks)

# ------------------------------------------------------------
# Payment audit readiness checks
# ------------------------------------------------------------

payments_enriched = payments_df.merge(
    invoices_df[
        [
            "invoice_id",
            "invoice_number",
            "invoice_amount",
            "invoice_date",
            "due_date",
            "payment_terms_days"
        ]
    ],
    on="invoice_id",
    how="left"
)

payments_enriched["days_between_invoice_and_payment"] = (
    payments_enriched["payment_date"] - payments_enriched["invoice_date"]
).dt.days

payments_enriched["days_late"] = (
    payments_enriched["payment_date"] - payments_enriched["due_date"]
).dt.days

payments_enriched["payment_amount_difference"] = (
    payments_enriched["payment_amount"] - payments_enriched["invoice_amount"]
)

payments_enriched["payment_amount_difference_pct"] = np.where(
    payments_enriched["invoice_amount"] != 0,
    payments_enriched["payment_amount_difference"] / payments_enriched["invoice_amount"] * 100,
    0
)

payment_before_invoice_date = payments_enriched["payment_date"] < payments_enriched["invoice_date"]
payment_after_due_date = payments_enriched["payment_date"] > payments_enriched["due_date"]
payment_more_than_30_days_late = payments_enriched["days_late"] > 30
payment_amount_mismatch = payments_enriched["payment_amount_difference_pct"].abs() > 5
weekend_payment = payments_enriched["payment_date"].dt.dayofweek.isin([5, 6])
cash_payment = payments_enriched["payment_method"] == "Cash"
failed_or_pending_payment = payments_enriched["payment_status"].isin(["Pending", "Failed"])

payment_audit_checks = pd.DataFrame({
    "Audit_Check": [
        "Payments before invoice date",
        "Payments after due date",
        "Payments more than 30 days late",
        "Payment amount mismatch > 5%",
        "Weekend payments",
        "Cash payments",
        "Pending or failed payments"
    ],
    "Number_of_Records": [
        payment_before_invoice_date.sum(),
        payment_after_due_date.sum(),
        payment_more_than_30_days_late.sum(),
        payment_amount_mismatch.sum(),
        weekend_payment.sum(),
        cash_payment.sum(),
        failed_or_pending_payment.sum()
    ]
})

payment_audit_checks["Percentage"] = (
    payment_audit_checks["Number_of_Records"] / len(payments_enriched) * 100
).round(2)

print("\nPayment audit checks:")
display(payment_audit_checks)

# ------------------------------------------------------------
# Supplier master data checks
# ------------------------------------------------------------

supplier_missing_tax_id = suppliers_df["tax_id"].isna()
supplier_missing_bank_account = suppliers_df["bank_account"].isna()
supplier_duplicate_bank_account = suppliers_df.duplicated(
    subset=["bank_account"],
    keep=False
)

supplier_master_checks = pd.DataFrame({
    "Audit_Check": [
        "Suppliers missing tax_id",
        "Suppliers missing bank account",
        "Suppliers with duplicate bank account"
    ],
    "Number_of_Records": [
        supplier_missing_tax_id.sum(),
        supplier_missing_bank_account.sum(),
        supplier_duplicate_bank_account.sum()
    ]
})

supplier_master_checks["Percentage"] = (
    supplier_master_checks["Number_of_Records"] / len(suppliers_df) * 100
).round(2)

print("\nSupplier master data checks:")
display(supplier_master_checks)

# ------------------------------------------------------------
# Supplier concentration quick check
# ------------------------------------------------------------

supplier_spend = (
    invoices_df
    .groupby("supplier_id", as_index=False)
    .agg(
        Invoice_Count=("invoice_id", "count"),
        Total_Invoice_Amount=("invoice_amount", "sum")
    )
    .sort_values("Total_Invoice_Amount", ascending=False)
)

supplier_spend["Spend_Share"] = (
    supplier_spend["Total_Invoice_Amount"] / supplier_spend["Total_Invoice_Amount"].sum() * 100
).round(2)

supplier_spend = supplier_spend.merge(
    suppliers_df[["supplier_id", "supplier_name", "supplier_category", "supplier_country"]],
    on="supplier_id",
    how="left"
)

print("\nTop 10 suppliers by invoice amount:")
display(supplier_spend.head(10))

# ------------------------------------------------------------
# Consolidated audit quality checks
# ------------------------------------------------------------

data_quality_audit_checks = pd.concat(
    [
        invoice_audit_checks.assign(Category="Invoice Checks"),
        payment_audit_checks.assign(Category="Payment Checks"),
        supplier_master_checks.assign(Category="Supplier Master Data")
    ],
    ignore_index=True
)

data_quality_audit_checks = data_quality_audit_checks[
    ["Category", "Audit_Check", "Number_of_Records", "Percentage"]
]

print("\nConsolidated audit quality checks:")
display(data_quality_audit_checks)

# ------------------------------------------------------------
# Save quality reports
# ------------------------------------------------------------

table_overview_path = REPORTS_DIR / "data_quality_table_overview.csv"
missing_values_path = REPORTS_DIR / "data_quality_missing_values.csv"
audit_checks_path = REPORTS_DIR / "data_quality_audit_checks.csv"
referential_integrity_path = REPORTS_DIR / "data_quality_referential_integrity.csv"
supplier_spend_path = REPORTS_DIR / "supplier_spend_concentration.csv"

table_overview_df.to_csv(table_overview_path, index=False)
missing_values_report.to_csv(missing_values_path, index=False)
data_quality_audit_checks.to_csv(audit_checks_path, index=False)
referential_integrity_report.to_csv(referential_integrity_path, index=False)
supplier_spend.to_csv(supplier_spend_path, index=False)

print("\nData quality reports saved successfully:")
print("-", table_overview_path)
print("-", missing_values_path)
print("-", audit_checks_path)
print("-", referential_integrity_path)
print("-", supplier_spend_path)

DATA QUALITY ASSESSMENT
Table overview:


,Table,Rows,Columns,Duplicate_Rows,Missing_Values_Total
0,suppliers,320,8,0,18
1,departments,12,4,0,0
2,purchase_orders,5200,7,0,0
3,invoices,8630,11,0,1496
4,payments,7918,8,0,0



Missing values report:


,Table,Column,Missing_Count,Missing_Percentage
23,invoices,po_id,1496,17.3300
6,suppliers,tax_id,18,5.6200



Data types report sample:


...


Top 10 suppliers by invoice amount:


,supplier_id,Invoice_Count,Total_Invoice_Amount,Spend_Share,supplier_name,supplier_category,supplier_country
0,SUP00009,325,"4,126,869.9800",3.7100,Supplier_0009,IT Services,Algeria
1,SUP00005,288,"4,000,165.6000",3.5900,Supplier_0005,Consulting,Algeria
2,SUP00002,282,"3,877,256.3800",3.4800,Supplier_0002,Training,Germany
3,SUP00010,274,"3,646,952.1000",3.2700,Supplier_0010,Facilities,Algeria
4,SUP00003,282,"3,549,871.8500",3.1900,Supplier_0003,Office Supplies,Turkey
5,SUP00004,266,"3,516,104.7200",3.1600,Supplier_0004,Professional Services,Turkey
6,SUP00007,264,"3,435,461.6100",3.0800,Supplier_0007,Telecom,Germany
7,SUP00006,278,"3,420,743.9500",3.0700,Supplier_0006,Telecom,Algeria
8,SUP00008,269,"3,250,205.0300",2.9200,Supplier_0008,Logistics,France
9,SUP00001,233,"2,431,526.3300",2.1800,Supplier_0001,Training,Turkey



Consolidated audit quality checks:


,Category,Audit_Check,Number_of_Records,Percentage
0,Invoice Checks,Invoices without purchase order,1496,17.3300
1,Invoice Checks,Zero or negative invoice amount,0,0.0000
2,Invoice Checks,Due date before invoice date,0,0.0000
3,Invoice Checks,Round amount invoices,426,4.9400
4,Invoice Checks,"High-value invoices >= 100,000",146,1.6900
5,Invoice Checks,Potential duplicate invoices,56,0.6500
6,Payment Checks,Payments before invoice date,260,3.2800
7,Payment Checks,Payments after due date,4906,61.9600
8,Payment Checks,Payments more than 30 days late,1227,15.5000
9,Payment Checks,Payment amount mismatch > 5%,7,0.0900



Data quality reports saved successfully:
- ..\reports\data_quality_table_overview.csv
- ..\reports\data_quality_missing_values.csv
- ..\reports\data_quality_audit_checks.csv
- ..\reports\data_quality_referential_integrity.csv
- ..\reports\supplier_spend_concentration.csv


# 4. Data Cleaning and Standardization

This section prepares clean and standardized financial audit tables.

The objective is not to remove suspicious records, but to preserve them and create audit flags that will be used later for risk scoring.

The cleaning process includes:

- standardizing missing purchase order values;
- cleaning supplier master data;
- creating missing tax ID flags;
- aggregating payment information by invoice;
- merging invoices with suppliers, departments, purchase orders and payments;
- creating a master audit table;
- creating preliminary audit flags;
- saving cleaned datasets for the next steps.

In audit analytics, anomalies should not be deleted. They should be identified, flagged and analyzed.

In [10]:
# ============================================================
# 4. DATA CLEANING AND STANDARDIZATION
# ============================================================

print("=" * 80)
print("DATA CLEANING AND STANDARDIZATION")
print("=" * 80)

# ------------------------------------------------------------
# Create clean copies
# ------------------------------------------------------------

suppliers_clean = suppliers_df.copy()
departments_clean = departments_df.copy()
purchase_orders_clean = purchase_orders_df.copy()
invoices_clean = invoices_df.copy()
payments_clean = payments_df.copy()

# ------------------------------------------------------------
# Standardize text columns
# ------------------------------------------------------------

def clean_text_columns(df):
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({
            "None": np.nan,
            "nan": np.nan,
            "NaT": np.nan,
            "": np.nan
        })
    return df

suppliers_clean = clean_text_columns(suppliers_clean)
departments_clean = clean_text_columns(departments_clean)
purchase_orders_clean = clean_text_columns(purchase_orders_clean)
invoices_clean = clean_text_columns(invoices_clean)
payments_clean = clean_text_columns(payments_clean)

# ------------------------------------------------------------
# Ensure date columns are datetime
# ------------------------------------------------------------

suppliers_clean["onboarding_date"] = pd.to_datetime(suppliers_clean["onboarding_date"])
purchase_orders_clean["po_date"] = pd.to_datetime(purchase_orders_clean["po_date"])

invoices_clean["invoice_date"] = pd.to_datetime(invoices_clean["invoice_date"])
invoices_clean["due_date"] = pd.to_datetime(invoices_clean["due_date"])

payments_clean["payment_date"] = pd.to_datetime(payments_clean["payment_date"])

# ------------------------------------------------------------
# Ensure numerical columns are numeric
# ------------------------------------------------------------

purchase_orders_clean["po_amount"] = pd.to_numeric(
    purchase_orders_clean["po_amount"],
    errors="coerce"
)

invoices_clean["invoice_amount"] = pd.to_numeric(
    invoices_clean["invoice_amount"],
    errors="coerce"
)

payments_clean["payment_amount"] = pd.to_numeric(
    payments_clean["payment_amount"],
    errors="coerce"
)

# ------------------------------------------------------------
# Supplier master data cleaning
# ------------------------------------------------------------

suppliers_clean["missing_tax_id_flag"] = suppliers_clean["tax_id"].isna().astype(int)
suppliers_clean["missing_bank_account_flag"] = suppliers_clean["bank_account"].isna().astype(int)

suppliers_clean["tax_id_clean"] = suppliers_clean["tax_id"].fillna("MISSING_TAX_ID")
suppliers_clean["supplier_country"] = suppliers_clean["supplier_country"].fillna("Unknown Country")
suppliers_clean["supplier_category"] = suppliers_clean["supplier_category"].fillna("Unknown Category")

# ------------------------------------------------------------
# Invoice cleaning flags
# ------------------------------------------------------------

invoices_clean["missing_po_flag"] = invoices_clean["po_id"].isna().astype(int)
invoices_clean["round_amount_flag"] = (
    invoices_clean["invoice_amount"] % 1000 == 0
).astype(int)

invoices_clean["high_value_invoice_flag"] = (
    invoices_clean["invoice_amount"] >= 100000
).astype(int)

invoices_clean["invoice_due_days"] = (
    invoices_clean["due_date"] - invoices_clean["invoice_date"]
).dt.days

invoices_clean["duplicate_invoice_flag"] = invoices_clean.duplicated(
    subset=["supplier_id", "invoice_number", "invoice_amount"],
    keep=False
).astype(int)

# ------------------------------------------------------------
# Payment cleaning and aggregation
# ------------------------------------------------------------

payments_clean["payment_weekday"] = payments_clean["payment_date"].dt.day_name()
payments_clean["weekend_payment_flag"] = (
    payments_clean["payment_date"].dt.dayofweek.isin([5, 6])
).astype(int)

payments_clean["cash_payment_flag"] = (
    payments_clean["payment_method"] == "Cash"
).astype(int)

payments_clean["pending_or_failed_payment_flag"] = (
    payments_clean["payment_status"].isin(["Pending", "Failed"])
).astype(int)

# In case multiple payments exist for one invoice, aggregate at invoice level
payment_summary = (
    payments_clean
    .groupby("invoice_id")
    .agg(
        payment_count=("payment_id", "count"),
        first_payment_date=("payment_date", "min"),
        last_payment_date=("payment_date", "max"),
        total_paid_amount=("payment_amount", "sum"),
        weekend_payment_flag=("weekend_payment_flag", "max"),
        cash_payment_flag=("cash_payment_flag", "max"),
        pending_or_failed_payment_flag=("pending_or_failed_payment_flag", "max"),
        payment_methods=("payment_method", lambda x: ", ".join(sorted(x.dropna().unique()))),
        payment_statuses=("payment_status", lambda x: ", ".join(sorted(x.dropna().unique())))
    )
    .reset_index()
)

# ------------------------------------------------------------
# Prepare PO information for audit comparison
# ------------------------------------------------------------

po_audit_columns = [
    "po_id",
    "supplier_id",
    "department_id",
    "po_date",
    "po_amount",
    "approval_status",
    "buyer"
]

po_for_merge = purchase_orders_clean[po_audit_columns].copy()

po_for_merge = po_for_merge.rename(columns={
    "supplier_id": "po_supplier_id",
    "department_id": "po_department_id",
    "po_date": "po_date",
    "po_amount": "po_amount",
    "approval_status": "po_approval_status",
    "buyer": "po_buyer"
})

# ------------------------------------------------------------
# Build master audit table
# ------------------------------------------------------------

audit_master_df = invoices_clean.merge(
    suppliers_clean[
        [
            "supplier_id",
            "supplier_name",
            "supplier_category",
            "supplier_country",
            "onboarding_date",
            "tax_id_clean",
            "missing_tax_id_flag",
            "bank_account",
            "missing_bank_account_flag"
        ]
    ],
    on="supplier_id",
    how="left"
)

audit_master_df = audit_master_df.merge(
    departments_clean[
        [
            "department_id",
            "department_name",
            "business_unit",
            "approval_threshold"
        ]
    ],
    on="department_id",
    how="left"
)

audit_master_df = audit_master_df.merge(
    po_for_merge,
    on="po_id",
    how="left"
)

audit_master_df = audit_master_df.merge(
    payment_summary,
    on="invoice_id",
    how="left"
)

# ------------------------------------------------------------
# Payment-related derived fields
# ------------------------------------------------------------

audit_master_df["paid_flag"] = audit_master_df["payment_count"].notna().astype(int)
audit_master_df["payment_count"] = audit_master_df["payment_count"].fillna(0).astype(int)
audit_master_df["total_paid_amount"] = audit_master_df["total_paid_amount"].fillna(0)

audit_master_df["days_to_payment"] = (
    audit_master_df["last_payment_date"] - audit_master_df["invoice_date"]
).dt.days

audit_master_df["days_late"] = (
    audit_master_df["last_payment_date"] - audit_master_df["due_date"]
).dt.days

audit_master_df["payment_before_invoice_flag"] = (
    audit_master_df["last_payment_date"] < audit_master_df["invoice_date"]
).fillna(False).astype(int)

audit_master_df["late_payment_flag"] = (
    audit_master_df["last_payment_date"] > audit_master_df["due_date"]
).fillna(False).astype(int)

audit_master_df["very_late_payment_flag"] = (
    audit_master_df["days_late"] > 30
).fillna(False).astype(int)

audit_master_df["payment_amount_difference"] = np.where(
    audit_master_df["paid_flag"] == 1,
    audit_master_df["total_paid_amount"] - audit_master_df["invoice_amount"],
    np.nan
)

audit_master_df["payment_amount_difference_pct"] = np.where(
    (audit_master_df["paid_flag"] == 1) & (audit_master_df["invoice_amount"] != 0),
    audit_master_df["payment_amount_difference"] / audit_master_df["invoice_amount"] * 100,
    np.nan
)

audit_master_df["payment_amount_mismatch_flag"] = (
    audit_master_df["payment_amount_difference_pct"].abs() > 5
).fillna(False).astype(int)

# ------------------------------------------------------------
# PO-related consistency flags
# ------------------------------------------------------------

audit_master_df["po_supplier_mismatch_flag"] = (
    (audit_master_df["missing_po_flag"] == 0) &
    (audit_master_df["supplier_id"] != audit_master_df["po_supplier_id"])
).fillna(False).astype(int)

audit_master_df["po_department_mismatch_flag"] = (
    (audit_master_df["missing_po_flag"] == 0) &
    (audit_master_df["department_id"] != audit_master_df["po_department_id"])
).fillna(False).astype(int)

audit_master_df["po_amount_difference"] = np.where(
    audit_master_df["missing_po_flag"] == 0,
    audit_master_df["invoice_amount"] - audit_master_df["po_amount"],
    np.nan
)

audit_master_df["po_amount_difference_pct"] = np.where(
    (audit_master_df["missing_po_flag"] == 0) & (audit_master_df["po_amount"] != 0),
    audit_master_df["po_amount_difference"] / audit_master_df["po_amount"] * 100,
    np.nan
)

audit_master_df["invoice_exceeds_po_amount_flag"] = (
    audit_master_df["po_amount_difference_pct"] > 10
).fillna(False).astype(int)

audit_master_df["po_not_approved_flag"] = (
    (audit_master_df["missing_po_flag"] == 0) &
    (audit_master_df["po_approval_status"] != "Approved")
).fillna(False).astype(int)

# ------------------------------------------------------------
# Approval threshold flag
# ------------------------------------------------------------

audit_master_df["above_department_threshold_flag"] = (
    audit_master_df["invoice_amount"] > audit_master_df["approval_threshold"]
).astype(int)

# ------------------------------------------------------------
# Time features
# ------------------------------------------------------------

audit_master_df["invoice_month"] = audit_master_df["invoice_date"].dt.month
audit_master_df["invoice_quarter"] = audit_master_df["invoice_date"].dt.quarter
audit_master_df["invoice_year"] = audit_master_df["invoice_date"].dt.year

# ------------------------------------------------------------
# Fill payment flags for unpaid invoices
# ------------------------------------------------------------

for col in [
    "weekend_payment_flag",
    "cash_payment_flag",
    "pending_or_failed_payment_flag"
]:
    audit_master_df[col] = audit_master_df[col].fillna(0).astype(int)

audit_master_df["payment_methods"] = audit_master_df["payment_methods"].fillna("Not Paid")
audit_master_df["payment_statuses"] = audit_master_df["payment_statuses"].fillna("Not Paid")

# ------------------------------------------------------------
# Standardized audit status
# ------------------------------------------------------------

def classify_payment_status(row):
    if row["paid_flag"] == 0:
        return "Unpaid"
    elif row["pending_or_failed_payment_flag"] == 1:
        return "Payment Issue"
    elif row["very_late_payment_flag"] == 1:
        return "Very Late Payment"
    elif row["late_payment_flag"] == 1:
        return "Late Payment"
    else:
        return "Paid On Time or Early"

audit_master_df["audit_payment_status"] = audit_master_df.apply(
    classify_payment_status,
    axis=1
)

# ------------------------------------------------------------
# Save cleaned datasets
# ------------------------------------------------------------

suppliers_clean_path = PROCESSED_DATA_DIR / "suppliers_clean.csv"
departments_clean_path = PROCESSED_DATA_DIR / "departments_clean.csv"
purchase_orders_clean_path = PROCESSED_DATA_DIR / "purchase_orders_clean.csv"
invoices_clean_path = PROCESSED_DATA_DIR / "invoices_clean.csv"
payments_clean_path = PROCESSED_DATA_DIR / "payments_clean.csv"
audit_master_path = PROCESSED_DATA_DIR / "audit_master.csv"

suppliers_clean.to_csv(suppliers_clean_path, index=False)
departments_clean.to_csv(departments_clean_path, index=False)
purchase_orders_clean.to_csv(purchase_orders_clean_path, index=False)
invoices_clean.to_csv(invoices_clean_path, index=False)
payments_clean.to_csv(payments_clean_path, index=False)
audit_master_df.to_csv(audit_master_path, index=False)

# ------------------------------------------------------------
# Output summary
# ------------------------------------------------------------

print("Data cleaning and standardization completed successfully")
print("-" * 80)
print("Suppliers clean shape       :", suppliers_clean.shape)
print("Departments clean shape     :", departments_clean.shape)
print("Purchase orders clean shape :", purchase_orders_clean.shape)
print("Invoices clean shape        :", invoices_clean.shape)
print("Payments clean shape        :", payments_clean.shape)
print("Audit master shape          :", audit_master_df.shape)

print("\nAudit master key columns preview:")
display(
    audit_master_df[
        [
            "invoice_id",
            "invoice_number",
            "supplier_id",
            "supplier_name",
            "department_name",
            "po_id",
            "invoice_amount",
            "total_paid_amount",
            "paid_flag",
            "days_late",
            "missing_po_flag",
            "duplicate_invoice_flag",
            "payment_before_invoice_flag",
            "late_payment_flag",
            "very_late_payment_flag",
            "payment_amount_mismatch_flag",
            "po_supplier_mismatch_flag",
            "po_department_mismatch_flag",
            "invoice_exceeds_po_amount_flag",
            "above_department_threshold_flag",
            "audit_payment_status"
        ]
    ].head()
)

print("\nPreliminary audit flags summary:")
preliminary_flags = [
    "missing_po_flag",
    "duplicate_invoice_flag",
    "round_amount_flag",
    "high_value_invoice_flag",
    "missing_tax_id_flag",
    "payment_before_invoice_flag",
    "late_payment_flag",
    "very_late_payment_flag",
    "payment_amount_mismatch_flag",
    "weekend_payment_flag",
    "cash_payment_flag",
    "pending_or_failed_payment_flag",
    "po_supplier_mismatch_flag",
    "po_department_mismatch_flag",
    "invoice_exceeds_po_amount_flag",
    "po_not_approved_flag",
    "above_department_threshold_flag"
]

preliminary_flags_summary = pd.DataFrame({
    "Audit_Flag": preliminary_flags,
    "Flagged_Records": [audit_master_df[col].sum() for col in preliminary_flags],
    "Percentage": [
        round(audit_master_df[col].mean() * 100, 2) for col in preliminary_flags
    ]
}).sort_values("Flagged_Records", ascending=False)

display(preliminary_flags_summary)

print("\nCleaned files saved successfully:")
print("-", suppliers_clean_path)
print("-", departments_clean_path)
print("-", purchase_orders_clean_path)
print("-", invoices_clean_path)
print("-", payments_clean_path)
print("-", audit_master_path)

DATA CLEANING AND STANDARDIZATION
Data cleaning and standardization completed successfully
--------------------------------------------------------------------------------
Suppliers clean shape       : (320, 11)
Departments clean shape     : (12, 4)
Purchase orders clean shape : (5200, 7)
Invoices clean shape        : (8630, 16)
Payments clean shape        : (7918, 12)
Audit master shape          : (8630, 62)

Audit master key columns preview:


,invoice_id,invoice_number,supplier_id,supplier_name,department_name,po_id,invoice_amount,total_paid_amount,paid_flag,days_late,missing_po_flag,duplicate_invoice_flag,payment_before_invoice_flag,late_payment_flag,very_late_payment_flag,payment_amount_mismatch_flag,po_supplier_mismatch_flag,po_department_mismatch_flag,invoice_exceeds_po_amount_flag,above_department_threshold_flag,audit_payment_status
0,INV0000001,FCT-634138,SUP00091,Supplier_0091,Risk Management,PO0002176,"1,082.2000","1,071.9700",1,67.0000,0,0,0,1,1,0,0,1,0,0,Very Late Payment
1,INV0000002,FCT-562234,SUP00057,Supplier_0057,Procurement,PO0003629,"17,142.8700","17,212.1800",1,10.0000,0,0,0,1,0,0,0,0,1,0,Late Payment
2,INV0000003,FCT-647518,SUP00011,Supplier_0011,Procurement,NaN,"1,411.5100","1,372.6200",1,-3.0000,1,0,0,0,0,0,0,0,0,0,Paid On Time or Early
3,INV0000004,FCT-802936,SUP00017,Supplier_0017,Marketing,NaN,"3,925.9000","3,979.9800",1,19.0000,1,0,0,1,0,0,0,0,0,0,Late Payment
4,INV0000005,FCT-449867,SUP00004,Supplier_0004,Risk Management,PO0000920,"15,000.0000","15,345.5000",1,32.0000,0,0,0,1,1,0,0,0,1,1,Very Late Payment



Preliminary audit flags summary:


,Audit_Flag,Flagged_Records,Percentage
6,late_payment_flag,4906,56.8500
14,invoice_exceeds_po_amount_flag,2893,33.5200
9,weekend_payment_flag,2459,28.4900
16,above_department_threshold_flag,2026,23.4800
0,missing_po_flag,1496,17.3300
7,very_late_payment_flag,1227,14.2200
15,po_not_approved_flag,792,9.1800
4,missing_tax_id_flag,502,5.8200
11,pending_or_failed_payment_flag,439,5.0900
2,round_amount_flag,426,4.9400



Cleaned files saved successfully:
- ..\data\processed\suppliers_clean.csv
- ..\data\processed\departments_clean.csv
- ..\data\processed\purchase_orders_clean.csv
- ..\data\processed\invoices_clean.csv
- ..\data\processed\payments_clean.csv
- ..\data\processed\audit_master.csv


## 2.1 Purchase Order Consistency Calibration

This step improves the realism of the synthetic audit dataset.

In a real procurement process, most invoices linked to a purchase order should match the same supplier and department as the purchase order. Only a small proportion should show supplier or department inconsistencies.

The calibration reassigns most purchase order links to consistent supplier-department combinations while keeping a controlled number of exceptions for audit testing.

In [8]:
# ============================================================
# 2.1 PURCHASE ORDER CONSISTENCY CALIBRATION - VERSION 2
# ============================================================

print("=" * 80)
print("PURCHASE ORDER CONSISTENCY CALIBRATION - VERSION 2")
print("=" * 80)

# ------------------------------------------------------------
# Objective:
# Most invoices linked to a PO should match the PO supplier and department.
# A controlled minority is intentionally kept inconsistent for audit testing.
# ------------------------------------------------------------

invoices_calibrated = invoices_df.copy()

po_reference = purchase_orders_df[
    ["po_id", "supplier_id", "department_id"]
].copy()

po_info = po_reference.set_index("po_id")

all_po_ids = po_reference["po_id"].tolist()

invoice_with_po_indices = invoices_calibrated[
    invoices_calibrated["po_id"].notna()
].index.tolist()

# ------------------------------------------------------------
# Step 1: Assign a valid PO to each invoice with PO
# Then align invoice supplier and department with the selected PO
# This creates a realistic base where most invoices are consistent.
# ------------------------------------------------------------

base_po_ids = rng.choice(
    all_po_ids,
    size=len(invoice_with_po_indices),
    replace=True
)

invoices_calibrated.loc[invoice_with_po_indices, "po_id"] = base_po_ids

for idx, po_id in zip(invoice_with_po_indices, base_po_ids):
    invoices_calibrated.loc[idx, "supplier_id"] = po_info.loc[po_id, "supplier_id"]
    invoices_calibrated.loc[idx, "department_id"] = po_info.loc[po_id, "department_id"]

# ------------------------------------------------------------
# Step 2: Create controlled PO mismatch exceptions
# ------------------------------------------------------------

supplier_mismatch_rate = 0.04
department_mismatch_rate = 0.05

n_with_po = len(invoice_with_po_indices)

supplier_mismatch_size = int(n_with_po * supplier_mismatch_rate)
department_mismatch_size = int(n_with_po * department_mismatch_rate)

supplier_mismatch_indices = set(
    rng.choice(
        invoice_with_po_indices,
        size=supplier_mismatch_size,
        replace=False
    )
)

remaining_indices = list(set(invoice_with_po_indices) - supplier_mismatch_indices)

department_mismatch_indices = set(
    rng.choice(
        remaining_indices,
        size=department_mismatch_size,
        replace=False
    )
)

# ------------------------------------------------------------
# Step 3: Force supplier mismatch while trying to keep department stable
# ------------------------------------------------------------

for idx in supplier_mismatch_indices:
    current_supplier = invoices_calibrated.loc[idx, "supplier_id"]
    current_department = invoices_calibrated.loc[idx, "department_id"]

    candidate_pos = po_reference[
        (po_reference["supplier_id"] != current_supplier) &
        (po_reference["department_id"] == current_department)
    ]

    if candidate_pos.empty:
        candidate_pos = po_reference[
            po_reference["supplier_id"] != current_supplier
        ]

    selected_po = candidate_pos.sample(
        n=1,
        random_state=int(rng.integers(0, 1_000_000))
    )["po_id"].iloc[0]

    invoices_calibrated.loc[idx, "po_id"] = selected_po

# ------------------------------------------------------------
# Step 4: Force department mismatch while trying to keep supplier stable
# ------------------------------------------------------------

for idx in department_mismatch_indices:
    current_supplier = invoices_calibrated.loc[idx, "supplier_id"]
    current_department = invoices_calibrated.loc[idx, "department_id"]

    candidate_pos = po_reference[
        (po_reference["supplier_id"] == current_supplier) &
        (po_reference["department_id"] != current_department)
    ]

    if candidate_pos.empty:
        candidate_pos = po_reference[
            po_reference["department_id"] != current_department
        ]

    selected_po = candidate_pos.sample(
        n=1,
        random_state=int(rng.integers(0, 1_000_000))
    )["po_id"].iloc[0]

    invoices_calibrated.loc[idx, "po_id"] = selected_po

# ------------------------------------------------------------
# Replace original invoices_df with calibrated version
# ------------------------------------------------------------

invoices_df = invoices_calibrated.copy()

# Save calibrated invoices
invoices_path = RAW_DATA_DIR / "invoices.csv"
invoices_df.to_csv(invoices_path, index=False)

# ------------------------------------------------------------
# Quick validation
# ------------------------------------------------------------

po_check = invoices_df.merge(
    purchase_orders_df[
        ["po_id", "supplier_id", "department_id"]
    ].rename(columns={
        "supplier_id": "po_supplier_id",
        "department_id": "po_department_id"
    }),
    on="po_id",
    how="left"
)

po_check_with_po = po_check[po_check["po_id"].notna()].copy()

supplier_mismatch_rate_after = (
    po_check_with_po["supplier_id"] != po_check_with_po["po_supplier_id"]
).mean() * 100

department_mismatch_rate_after = (
    po_check_with_po["department_id"] != po_check_with_po["po_department_id"]
).mean() * 100

print("PO consistency calibration V2 completed successfully")
print("Invoices shape:", invoices_df.shape)
print("Invoices with PO:", po_check_with_po.shape[0])
print("Supplier mismatch rate after calibration:", round(supplier_mismatch_rate_after, 2), "%")
print("Department mismatch rate after calibration:", round(department_mismatch_rate_after, 2), "%")
print("Calibrated invoices saved at:", invoices_path)

PURCHASE ORDER CONSISTENCY CALIBRATION - VERSION 2
PO consistency calibration V2 completed successfully
Invoices shape: (8630, 11)
Invoices with PO: 7134
Supplier mismatch rate after calibration: 3.99 %
Department mismatch rate after calibration: 4.99 %
Calibrated invoices saved at: ..\data\raw\invoices.csv


# 5. Audit Rules Engine

This section builds an automated audit rules engine.

The objective is to convert audit flags into structured control rules that can be used to identify risky invoices, payment exceptions, supplier master data issues and internal control weaknesses.

Each audit rule is assigned:

- a rule code;
- a rule name;
- a risk category;
- a severity level;
- a numerical risk weight.

The rules engine produces:

- transaction-level audit risk scores;
- audit risk levels;
- number of triggered rules per invoice;
- primary risk driver;
- audit exception tables;
- rule-level and category-level summaries.

This approach is similar to audit analytics procedures used in internal audit, external audit and risk advisory engagements.

In [11]:
# ============================================================
# 5. AUDIT RULES ENGINE
# ============================================================

print("=" * 80)
print("AUDIT RULES ENGINE")
print("=" * 80)

# ------------------------------------------------------------
# Create a scored audit dataset
# ------------------------------------------------------------

audit_scored_df = audit_master_df.copy()

# ------------------------------------------------------------
# Additional duplicate detection rules
# ------------------------------------------------------------

# Strict duplicate: same supplier, invoice number and amount
audit_scored_df["duplicate_invoice_strict_flag"] = audit_scored_df.duplicated(
    subset=["supplier_id", "invoice_number", "invoice_amount"],
    keep=False
).astype(int)

# Broader duplicate: same invoice number and amount, regardless supplier
audit_scored_df["duplicate_invoice_number_amount_flag"] = audit_scored_df.duplicated(
    subset=["invoice_number", "invoice_amount"],
    keep=False
).astype(int)

# Same supplier and amount close in time
audit_scored_df = audit_scored_df.sort_values(
    ["supplier_id", "invoice_amount", "invoice_date"]
).copy()

audit_scored_df["previous_invoice_date_same_supplier_amount"] = (
    audit_scored_df
    .groupby(["supplier_id", "invoice_amount"])["invoice_date"]
    .shift(1)
)

audit_scored_df["days_since_previous_same_amount"] = (
    audit_scored_df["invoice_date"] -
    audit_scored_df["previous_invoice_date_same_supplier_amount"]
).dt.days

audit_scored_df["same_supplier_same_amount_close_date_flag"] = (
    audit_scored_df["days_since_previous_same_amount"].between(0, 7)
).fillna(False).astype(int)

# Final duplicate risk flag
audit_scored_df["audit_duplicate_risk_flag"] = (
    (
        audit_scored_df["duplicate_invoice_strict_flag"] == 1
    ) |
    (
        audit_scored_df["duplicate_invoice_number_amount_flag"] == 1
    ) |
    (
        audit_scored_df["same_supplier_same_amount_close_date_flag"] == 1
    )
).astype(int)

# Restore original order
audit_scored_df = audit_scored_df.sort_values("invoice_id").reset_index(drop=True)

# ------------------------------------------------------------
# Ensure all base audit flag columns exist
# ------------------------------------------------------------

required_flag_columns = [
    "missing_po_flag",
    "round_amount_flag",
    "high_value_invoice_flag",
    "missing_tax_id_flag",
    "payment_before_invoice_flag",
    "late_payment_flag",
    "very_late_payment_flag",
    "payment_amount_mismatch_flag",
    "weekend_payment_flag",
    "cash_payment_flag",
    "pending_or_failed_payment_flag",
    "po_supplier_mismatch_flag",
    "po_department_mismatch_flag",
    "invoice_exceeds_po_amount_flag",
    "po_not_approved_flag",
    "above_department_threshold_flag",
    "audit_duplicate_risk_flag"
]

for col in required_flag_columns:
    if col not in audit_scored_df.columns:
        audit_scored_df[col] = 0
    
    audit_scored_df[col] = audit_scored_df[col].fillna(0).astype(int)

# ------------------------------------------------------------
# Audit rules configuration
# ------------------------------------------------------------

audit_rules_config = pd.DataFrame([
    {
        "Rule_Code": "R01",
        "Flag_Column": "missing_po_flag",
        "Rule_Name": "Invoice without purchase order",
        "Risk_Category": "Procurement Control",
        "Severity": "High",
        "Risk_Weight": 12,
        "Rule_Description": "Invoice is not linked to a purchase order."
    },
    {
        "Rule_Code": "R02",
        "Flag_Column": "audit_duplicate_risk_flag",
        "Rule_Name": "Potential duplicate invoice",
        "Risk_Category": "Duplicate Payment Risk",
        "Severity": "Critical",
        "Risk_Weight": 18,
        "Rule_Description": "Invoice appears to be duplicated or very similar to another invoice."
    },
    {
        "Rule_Code": "R03",
        "Flag_Column": "round_amount_flag",
        "Rule_Name": "Round amount invoice",
        "Risk_Category": "Amount Anomaly",
        "Severity": "Medium",
        "Risk_Weight": 6,
        "Rule_Description": "Invoice amount is a round amount, which may require additional review."
    },
    {
        "Rule_Code": "R04",
        "Flag_Column": "high_value_invoice_flag",
        "Rule_Name": "High-value invoice",
        "Risk_Category": "Financial Exposure",
        "Severity": "High",
        "Risk_Weight": 12,
        "Rule_Description": "Invoice amount is greater than or equal to 100,000."
    },
    {
        "Rule_Code": "R05",
        "Flag_Column": "missing_tax_id_flag",
        "Rule_Name": "Supplier missing tax ID",
        "Risk_Category": "Supplier Master Data",
        "Severity": "High",
        "Risk_Weight": 10,
        "Rule_Description": "Supplier master data is incomplete because the tax ID is missing."
    },
    {
        "Rule_Code": "R06",
        "Flag_Column": "payment_before_invoice_flag",
        "Rule_Name": "Payment before invoice date",
        "Risk_Category": "Payment Timing",
        "Severity": "Critical",
        "Risk_Weight": 18,
        "Rule_Description": "Payment date occurs before the invoice date."
    },
    {
        "Rule_Code": "R07",
        "Flag_Column": "very_late_payment_flag",
        "Rule_Name": "Very late payment",
        "Risk_Category": "Payment Timing",
        "Severity": "High",
        "Risk_Weight": 10,
        "Rule_Description": "Payment is more than 30 days after the due date."
    },
    {
        "Rule_Code": "R08",
        "Flag_Column": "payment_amount_mismatch_flag",
        "Rule_Name": "Payment amount mismatch",
        "Risk_Category": "Payment Accuracy",
        "Severity": "Critical",
        "Risk_Weight": 18,
        "Rule_Description": "Paid amount differs from invoice amount by more than 5%."
    },
    {
        "Rule_Code": "R09",
        "Flag_Column": "weekend_payment_flag",
        "Rule_Name": "Weekend payment",
        "Risk_Category": "Payment Behavior",
        "Severity": "Medium",
        "Risk_Weight": 5,
        "Rule_Description": "Payment was executed during the weekend."
    },
    {
        "Rule_Code": "R10",
        "Flag_Column": "cash_payment_flag",
        "Rule_Name": "Cash payment",
        "Risk_Category": "Payment Method",
        "Severity": "High",
        "Risk_Weight": 10,
        "Rule_Description": "Payment was made using cash."
    },
    {
        "Rule_Code": "R11",
        "Flag_Column": "pending_or_failed_payment_flag",
        "Rule_Name": "Pending or failed payment",
        "Risk_Category": "Payment Status",
        "Severity": "Medium",
        "Risk_Weight": 7,
        "Rule_Description": "Payment status is pending or failed."
    },
    {
        "Rule_Code": "R12",
        "Flag_Column": "po_supplier_mismatch_flag",
        "Rule_Name": "PO supplier mismatch",
        "Risk_Category": "Procurement Control",
        "Severity": "Critical",
        "Risk_Weight": 20,
        "Rule_Description": "Invoice supplier does not match the supplier linked to the purchase order."
    },
    {
        "Rule_Code": "R13",
        "Flag_Column": "po_department_mismatch_flag",
        "Rule_Name": "PO department mismatch",
        "Risk_Category": "Procurement Control",
        "Severity": "High",
        "Risk_Weight": 12,
        "Rule_Description": "Invoice department does not match the department linked to the purchase order."
    },
    {
        "Rule_Code": "R14",
        "Flag_Column": "invoice_exceeds_po_amount_flag",
        "Rule_Name": "Invoice exceeds PO amount",
        "Risk_Category": "Procurement Control",
        "Severity": "High",
        "Risk_Weight": 10,
        "Rule_Description": "Invoice amount exceeds purchase order amount by more than 10%."
    },
    {
        "Rule_Code": "R15",
        "Flag_Column": "po_not_approved_flag",
        "Rule_Name": "PO not approved",
        "Risk_Category": "Approval Control",
        "Severity": "High",
        "Risk_Weight": 12,
        "Rule_Description": "Invoice is linked to a purchase order that is not approved."
    },
    {
        "Rule_Code": "R16",
        "Flag_Column": "above_department_threshold_flag",
        "Rule_Name": "Above department approval threshold",
        "Risk_Category": "Approval Control",
        "Severity": "Medium",
        "Risk_Weight": 6,
        "Rule_Description": "Invoice amount exceeds the department approval threshold."
    }
])

# ------------------------------------------------------------
# Apply audit rules and calculate risk score
# ------------------------------------------------------------

audit_scored_df["audit_raw_risk_score"] = 0
audit_scored_df["audit_rule_count"] = 0

for _, rule in audit_rules_config.iterrows():
    flag_col = rule["Flag_Column"]
    score_col = f"{rule['Rule_Code']}_{flag_col}"
    
    audit_scored_df[score_col] = audit_scored_df[flag_col] * rule["Risk_Weight"]
    audit_scored_df["audit_raw_risk_score"] += audit_scored_df[score_col]
    audit_scored_df["audit_rule_count"] += audit_scored_df[flag_col]

max_possible_score = audit_rules_config["Risk_Weight"].sum()

audit_scored_df["audit_risk_score"] = (
    audit_scored_df["audit_raw_risk_score"] / max_possible_score * 100
).round(2)

# ------------------------------------------------------------
# Risk level classification
# ------------------------------------------------------------

def classify_audit_risk(score):
    if score >= 45:
        return "Critical Risk"
    elif score >= 30:
        return "High Risk"
    elif score >= 15:
        return "Medium Risk"
    elif score > 0:
        return "Low Risk"
    else:
        return "No Exception"

audit_scored_df["audit_risk_level"] = audit_scored_df["audit_risk_score"].apply(
    classify_audit_risk
)

# ------------------------------------------------------------
# Primary risk driver
# ------------------------------------------------------------

rule_lookup = audit_rules_config.set_index("Flag_Column").to_dict("index")

def get_primary_risk_driver(row):
    triggered_rules = []
    
    for flag_col, rule_info in rule_lookup.items():
        if row.get(flag_col, 0) == 1:
            triggered_rules.append(
                (
                    rule_info["Risk_Weight"],
                    rule_info["Rule_Code"],
                    rule_info["Rule_Name"]
                )
            )
    
    if not triggered_rules:
        return "No Exception"
    
    triggered_rules = sorted(triggered_rules, reverse=True)
    return f"{triggered_rules[0][1]} - {triggered_rules[0][2]}"

audit_scored_df["primary_risk_driver"] = audit_scored_df.apply(
    get_primary_risk_driver,
    axis=1
)

# ------------------------------------------------------------
# Create audit exceptions table
# ------------------------------------------------------------

exception_records = []

for _, rule in audit_rules_config.iterrows():
    flag_col = rule["Flag_Column"]
    
    triggered = audit_scored_df[audit_scored_df[flag_col] == 1].copy()
    
    if len(triggered) > 0:
        temp = triggered[
            [
                "invoice_id",
                "invoice_number",
                "supplier_id",
                "supplier_name",
                "department_id",
                "department_name",
                "invoice_date",
                "invoice_amount",
                "paid_flag",
                "audit_risk_score",
                "audit_risk_level"
            ]
        ].copy()
        
        temp["Rule_Code"] = rule["Rule_Code"]
        temp["Rule_Name"] = rule["Rule_Name"]
        temp["Risk_Category"] = rule["Risk_Category"]
        temp["Severity"] = rule["Severity"]
        temp["Risk_Weight"] = rule["Risk_Weight"]
        temp["Rule_Description"] = rule["Rule_Description"]
        
        exception_records.append(temp)

audit_exceptions_df = pd.concat(exception_records, ignore_index=True)

# ------------------------------------------------------------
# Rule-level summary
# ------------------------------------------------------------

audit_rules_summary = []

for _, rule in audit_rules_config.iterrows():
    flag_col = rule["Flag_Column"]
    count = audit_scored_df[flag_col].sum()
    
    audit_rules_summary.append({
        "Rule_Code": rule["Rule_Code"],
        "Rule_Name": rule["Rule_Name"],
        "Risk_Category": rule["Risk_Category"],
        "Severity": rule["Severity"],
        "Risk_Weight": rule["Risk_Weight"],
        "Flagged_Invoices": int(count),
        "Flagged_Percentage": round(count / len(audit_scored_df) * 100, 2),
        "Total_Invoice_Amount": round(
            audit_scored_df.loc[audit_scored_df[flag_col] == 1, "invoice_amount"].sum(),
            2
        )
    })

audit_rules_summary_df = pd.DataFrame(audit_rules_summary)
audit_rules_summary_df = audit_rules_summary_df.sort_values(
    ["Risk_Weight", "Flagged_Invoices"],
    ascending=False
)

# ------------------------------------------------------------
# Category-level summary
# ------------------------------------------------------------

audit_category_summary = (
    audit_exceptions_df
    .groupby("Risk_Category")
    .agg(
        Exceptions=("invoice_id", "count"),
        Unique_Invoices=("invoice_id", "nunique"),
        Total_Invoice_Amount=("invoice_amount", "sum"),
        Average_Risk_Score=("audit_risk_score", "mean")
    )
    .reset_index()
)

audit_category_summary["Total_Invoice_Amount"] = audit_category_summary["Total_Invoice_Amount"].round(2)
audit_category_summary["Average_Risk_Score"] = audit_category_summary["Average_Risk_Score"].round(2)

audit_category_summary = audit_category_summary.sort_values(
    "Exceptions",
    ascending=False
)

# ------------------------------------------------------------
# Risk level distribution
# ------------------------------------------------------------

risk_level_distribution = (
    audit_scored_df["audit_risk_level"]
    .value_counts()
    .reset_index()
)

risk_level_distribution.columns = ["Audit_Risk_Level", "Invoices"]

risk_level_distribution["Percentage"] = (
    risk_level_distribution["Invoices"] / len(audit_scored_df) * 100
).round(2)

# ------------------------------------------------------------
# Top high-risk invoices
# ------------------------------------------------------------

top_high_risk_invoices = (
    audit_scored_df
    .sort_values(
        ["audit_risk_score", "invoice_amount"],
        ascending=False
    )
    [
        [
            "invoice_id",
            "invoice_number",
            "supplier_name",
            "department_name",
            "invoice_date",
            "invoice_amount",
            "paid_flag",
            "audit_rule_count",
            "audit_risk_score",
            "audit_risk_level",
            "primary_risk_driver"
        ]
    ]
    .head(20)
)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

audit_scored_path = PROCESSED_DATA_DIR / "audit_scored_transactions.csv"
audit_exceptions_path = REPORTS_DIR / "audit_exceptions.csv"
audit_rules_summary_path = REPORTS_DIR / "audit_rules_summary.csv"
audit_category_summary_path = REPORTS_DIR / "audit_category_summary.csv"
risk_level_distribution_path = REPORTS_DIR / "risk_level_distribution.csv"
top_high_risk_invoices_path = REPORTS_DIR / "top_high_risk_invoices.csv"

audit_scored_df.to_csv(audit_scored_path, index=False)
audit_exceptions_df.to_csv(audit_exceptions_path, index=False)
audit_rules_summary_df.to_csv(audit_rules_summary_path, index=False)
audit_category_summary.to_csv(audit_category_summary_path, index=False)
risk_level_distribution.to_csv(risk_level_distribution_path, index=False)
top_high_risk_invoices.to_csv(top_high_risk_invoices_path, index=False)

# ------------------------------------------------------------
# Display outputs
# ------------------------------------------------------------

print("Audit rules engine completed successfully")
print("Audit scored dataset shape:", audit_scored_df.shape)
print("Audit exceptions table shape:", audit_exceptions_df.shape)

print("\nAudit rules summary:")
display(audit_rules_summary_df)

print("\nRisk level distribution:")
display(risk_level_distribution)

print("\nAudit category summary:")
display(audit_category_summary)

print("\nTop 20 high-risk invoices:")
display(top_high_risk_invoices)

print("\nFiles saved successfully:")
print("-", audit_scored_path)
print("-", audit_exceptions_path)
print("-", audit_rules_summary_path)
print("-", audit_category_summary_path)
print("-", risk_level_distribution_path)
print("-", top_high_risk_invoices_path)

AUDIT RULES ENGINE
Audit rules engine completed successfully
Audit scored dataset shape: (8630, 89)
Audit exceptions table shape: (13750, 17)

Audit rules summary:


,Rule_Code,Rule_Name,Risk_Category,Severity,Risk_Weight,Flagged_Invoices,Flagged_Percentage,Total_Invoice_Amount
11,R12,PO supplier mismatch,Procurement Control,Critical,20,285,3.3000,"3,449,634.8500"
1,R02,Potential duplicate invoice,Duplicate Payment Risk,Critical,18,265,3.0700,"3,780,566.0000"
5,R06,Payment before invoice date,Payment Timing,Critical,18,260,3.0100,"3,444,329.0000"
7,R08,Payment amount mismatch,Payment Accuracy,Critical,18,7,0.0800,"87,189.9200"
0,R01,Invoice without purchase order,Procurement Control,High,12,1496,17.3300,"20,623,340.2700"
14,R15,PO not approved,Approval Control,High,12,792,9.1800,"10,887,758.5900"
12,R13,PO department mismatch,Procurement Control,High,12,356,4.1300,"4,624,716.2500"
3,R04,High-value invoice,Financial Exposure,High,12,146,1.6900,"18,479,158.0500"
13,R14,Invoice exceeds PO amount,Procurement Control,High,10,2893,33.5200,"67,685,330.8200"
6,R07,Very late payment,Payment Timing,High,10,1227,14.2200,"15,038,911.8700"



Risk level distribution:


,Audit_Risk_Level,Invoices,Percentage
0,Low Risk,5970,69.1800
1,No Exception,1514,17.5400
2,Medium Risk,1125,13.0400
3,High Risk,21,0.2400



Audit category summary:


,Risk_Category,Exceptions,Unique_Invoices,Total_Invoice_Amount,Average_Risk_Score
9,Procurement Control,5030,4809,"96,383,022.1900",11.4600
1,Approval Control,2818,2627,"73,725,588.8600",12.5200
5,Payment Behavior,2459,2459,"32,169,411.0700",9.9500
8,Payment Timing,1487,1487,"18,483,240.8700",13.2100
10,Supplier Master Data,502,502,"6,189,450.5500",12.9300
7,Payment Status,439,439,"5,596,606.0600",11.3200
0,Amount Anomaly,426,426,"14,367,000.0000",14.2900
2,Duplicate Payment Risk,265,265,"3,780,566.0000",16.7800
6,Payment Method,171,171,"2,130,617.8500",13.8300
3,Financial Exposure,146,146,"18,479,158.0500",20.9600



Top 20 high-risk invoices:


,invoice_id,invoice_number,supplier_name,department_name,invoice_date,invoice_amount,paid_flag,audit_rule_count,audit_risk_score,audit_risk_level,primary_risk_driver
1560,INV0001561,FCT-405039,Supplier_0225,Operations,2024-04-09,"156,405.6100",1,6,38.1700,High Risk,R12 - PO supplier mismatch
4893,INV0004894,FCT-493456,Supplier_0089,Human Resources,2024-02-14,"1,576.3700",1,5,34.9500,High Risk,R12 - PO supplier mismatch
1246,INV0001247,FCT-449128,Supplier_0003,Legal,2024-01-27,"100,000.0000",1,6,34.4100,High Risk,R12 - PO supplier mismatch
8473,INV0008474,FCT-328934,Supplier_0021,Risk Management,2024-09-13,"27,036.3500",1,5,34.4100,High Risk,R12 - PO supplier mismatch
6515,INV0006516,FCT-701982,Supplier_0074,Marketing,2024-06-19,"138,721.0300",1,6,33.8700,High Risk,R06 - Payment before invoice date
3951,INV0003952,FCT-843469,Supplier_0233,Operations,2024-10-20,"100,000.0000",1,7,33.8700,High Risk,R15 - PO not approved
1655,INV0001656,FCT-618610,Supplier_0001,Finance,2024-07-10,"3,104.2800",1,5,33.8700,High Risk,R12 - PO supplier mismatch
7927,INV0007928,FCT-704946,Supplier_0007,Logistics,2024-06-14,"100,000.0000",1,7,32.8000,High Risk,R13 - PO department mismatch
671,INV0000672,FCT-510874,Supplier_0139,Logistics,2024-01-11,"6,015.6900",1,4,32.2600,High Risk,R12 - PO supplier mismatch
8034,INV0008035,FCT-146312,Supplier_0301,Sales,2024-11-24,"28,373.7300",1,6,31.7200,High Risk,R02 - Potential duplicate invoice



Files saved successfully:
- ..\data\processed\audit_scored_transactions.csv
- ..\reports\audit_exceptions.csv
- ..\reports\audit_rules_summary.csv
- ..\reports\audit_category_summary.csv
- ..\reports\risk_level_distribution.csv
- ..\reports\top_high_risk_invoices.csv


# 6. Supplier Risk Scoring

This section creates a supplier-level risk scoring model.

The objective is to aggregate transaction-level audit findings into supplier-level risk indicators.

The supplier risk score considers:

- total invoice amount;
- supplier spend concentration;
- number of audit exceptions;
- average transaction risk score;
- high-risk invoice share;
- critical audit rule triggers;
- procurement control exceptions;
- payment control exceptions;
- supplier master data issues.

The final output is a supplier risk ranking that can help audit teams prioritize supplier reviews and focus audit procedures on the highest-risk third parties.

In [12]:
# ============================================================
# 6. SUPPLIER RISK SCORING
# ============================================================

print("=" * 80)
print("SUPPLIER RISK SCORING")
print("=" * 80)

# ------------------------------------------------------------
# Helper function for scaling
# ------------------------------------------------------------

def min_max_scale(series):
    min_value = series.min()
    max_value = series.max()
    
    if max_value == min_value:
        return pd.Series(0, index=series.index)
    
    return (series - min_value) / (max_value - min_value)

# ------------------------------------------------------------
# Define risk groups
# ------------------------------------------------------------

critical_rule_flags = [
    "po_supplier_mismatch_flag",
    "audit_duplicate_risk_flag",
    "payment_before_invoice_flag",
    "payment_amount_mismatch_flag"
]

procurement_rule_flags = [
    "missing_po_flag",
    "po_supplier_mismatch_flag",
    "po_department_mismatch_flag",
    "invoice_exceeds_po_amount_flag",
    "po_not_approved_flag"
]

payment_rule_flags = [
    "payment_before_invoice_flag",
    "very_late_payment_flag",
    "payment_amount_mismatch_flag",
    "weekend_payment_flag",
    "cash_payment_flag",
    "pending_or_failed_payment_flag"
]

# Make sure columns exist
for col in critical_rule_flags + procurement_rule_flags + payment_rule_flags:
    if col not in audit_scored_df.columns:
        audit_scored_df[col] = 0

# ------------------------------------------------------------
# Supplier-level aggregation
# ------------------------------------------------------------

supplier_risk_summary = (
    audit_scored_df
    .groupby(["supplier_id", "supplier_name", "supplier_category", "supplier_country"])
    .agg(
        Invoice_Count=("invoice_id", "count"),
        Total_Invoice_Amount=("invoice_amount", "sum"),
        Average_Invoice_Amount=("invoice_amount", "mean"),
        Paid_Invoices=("paid_flag", "sum"),
        Average_Audit_Risk_Score=("audit_risk_score", "mean"),
        Max_Audit_Risk_Score=("audit_risk_score", "max"),
        Average_Rule_Count=("audit_rule_count", "mean"),
        Exception_Invoices=("audit_rule_count", lambda x: (x > 0).sum()),
        High_Risk_Invoices=("audit_risk_level", lambda x: x.isin(["High Risk", "Critical Risk"]).sum()),
        Medium_Risk_Invoices=("audit_risk_level", lambda x: (x == "Medium Risk").sum()),
        Missing_PO_Count=("missing_po_flag", "sum"),
        Duplicate_Risk_Count=("audit_duplicate_risk_flag", "sum"),
        Payment_Before_Invoice_Count=("payment_before_invoice_flag", "sum"),
        Very_Late_Payment_Count=("very_late_payment_flag", "sum"),
        Weekend_Payment_Count=("weekend_payment_flag", "sum"),
        Cash_Payment_Count=("cash_payment_flag", "sum"),
        PO_Supplier_Mismatch_Count=("po_supplier_mismatch_flag", "sum"),
        PO_Department_Mismatch_Count=("po_department_mismatch_flag", "sum"),
        Invoice_Exceeds_PO_Count=("invoice_exceeds_po_amount_flag", "sum"),
        PO_Not_Approved_Count=("po_not_approved_flag", "sum"),
        Missing_Tax_ID_Flag=("missing_tax_id_flag", "max")
    )
    .reset_index()
)

# ------------------------------------------------------------
# Additional supplier risk indicators
# ------------------------------------------------------------

supplier_risk_summary["Spend_Share"] = (
    supplier_risk_summary["Total_Invoice_Amount"] /
    supplier_risk_summary["Total_Invoice_Amount"].sum() * 100
)

supplier_risk_summary["Exception_Rate"] = (
    supplier_risk_summary["Exception_Invoices"] /
    supplier_risk_summary["Invoice_Count"] * 100
)

supplier_risk_summary["High_Risk_Invoice_Rate"] = (
    supplier_risk_summary["High_Risk_Invoices"] /
    supplier_risk_summary["Invoice_Count"] * 100
)

supplier_risk_summary["Medium_Risk_Invoice_Rate"] = (
    supplier_risk_summary["Medium_Risk_Invoices"] /
    supplier_risk_summary["Invoice_Count"] * 100
)

supplier_risk_summary["Critical_Rule_Triggers"] = (
    audit_scored_df
    .assign(critical_rule_trigger_count=audit_scored_df[critical_rule_flags].sum(axis=1))
    .groupby("supplier_id")["critical_rule_trigger_count"]
    .sum()
    .reindex(supplier_risk_summary["supplier_id"])
    .values
)

supplier_risk_summary["Procurement_Exception_Count"] = (
    audit_scored_df
    .assign(procurement_exception_count=audit_scored_df[procurement_rule_flags].sum(axis=1))
    .groupby("supplier_id")["procurement_exception_count"]
    .sum()
    .reindex(supplier_risk_summary["supplier_id"])
    .values
)

supplier_risk_summary["Payment_Exception_Count"] = (
    audit_scored_df
    .assign(payment_exception_count=audit_scored_df[payment_rule_flags].sum(axis=1))
    .groupby("supplier_id")["payment_exception_count"]
    .sum()
    .reindex(supplier_risk_summary["supplier_id"])
    .values
)

supplier_risk_summary["Procurement_Exception_Rate"] = (
    supplier_risk_summary["Procurement_Exception_Count"] /
    supplier_risk_summary["Invoice_Count"] * 100
)

supplier_risk_summary["Payment_Exception_Rate"] = (
    supplier_risk_summary["Payment_Exception_Count"] /
    supplier_risk_summary["Invoice_Count"] * 100
)

# ------------------------------------------------------------
# Normalized risk components
# ------------------------------------------------------------

supplier_risk_summary["avg_risk_score_scaled"] = min_max_scale(
    supplier_risk_summary["Average_Audit_Risk_Score"]
)

supplier_risk_summary["exception_rate_scaled"] = min_max_scale(
    supplier_risk_summary["Exception_Rate"]
)

supplier_risk_summary["high_risk_rate_scaled"] = min_max_scale(
    supplier_risk_summary["High_Risk_Invoice_Rate"]
)

supplier_risk_summary["critical_triggers_scaled"] = min_max_scale(
    supplier_risk_summary["Critical_Rule_Triggers"]
)

supplier_risk_summary["spend_share_scaled"] = min_max_scale(
    supplier_risk_summary["Spend_Share"]
)

supplier_risk_summary["procurement_exception_scaled"] = min_max_scale(
    supplier_risk_summary["Procurement_Exception_Rate"]
)

supplier_risk_summary["payment_exception_scaled"] = min_max_scale(
    supplier_risk_summary["Payment_Exception_Rate"]
)

# ------------------------------------------------------------
# Supplier risk score
# ------------------------------------------------------------

supplier_risk_summary["supplier_risk_score"] = (
    0.25 * supplier_risk_summary["avg_risk_score_scaled"] +
    0.15 * supplier_risk_summary["exception_rate_scaled"] +
    0.15 * supplier_risk_summary["high_risk_rate_scaled"] +
    0.15 * supplier_risk_summary["critical_triggers_scaled"] +
    0.10 * supplier_risk_summary["spend_share_scaled"] +
    0.10 * supplier_risk_summary["procurement_exception_scaled"] +
    0.07 * supplier_risk_summary["payment_exception_scaled"] +
    0.03 * supplier_risk_summary["Missing_Tax_ID_Flag"]
) * 100

supplier_risk_summary["supplier_risk_score"] = supplier_risk_summary["supplier_risk_score"].round(2)

# ------------------------------------------------------------
# Supplier risk classification
# ------------------------------------------------------------

def classify_supplier_risk(score):
    if score >= 70:
        return "Critical Supplier Risk"
    elif score >= 50:
        return "High Supplier Risk"
    elif score >= 30:
        return "Medium Supplier Risk"
    elif score > 0:
        return "Low Supplier Risk"
    else:
        return "No Supplier Risk"

supplier_risk_summary["supplier_risk_level"] = supplier_risk_summary["supplier_risk_score"].apply(
    classify_supplier_risk
)

# ------------------------------------------------------------
# Supplier risk driver
# ------------------------------------------------------------

def identify_supplier_risk_driver(row):
    drivers = {
        "High average transaction risk": row["avg_risk_score_scaled"],
        "High exception rate": row["exception_rate_scaled"],
        "High-risk invoices": row["high_risk_rate_scaled"],
        "Critical audit rule triggers": row["critical_triggers_scaled"],
        "Spend concentration": row["spend_share_scaled"],
        "Procurement control exceptions": row["procurement_exception_scaled"],
        "Payment control exceptions": row["payment_exception_scaled"],
        "Missing tax ID": row["Missing_Tax_ID_Flag"]
    }
    
    return max(drivers, key=drivers.get)

supplier_risk_summary["primary_supplier_risk_driver"] = supplier_risk_summary.apply(
    identify_supplier_risk_driver,
    axis=1
)

# ------------------------------------------------------------
# Audit recommendation by supplier
# ------------------------------------------------------------

def generate_supplier_recommendation(row):
    risk_level = row["supplier_risk_level"]
    driver = row["primary_supplier_risk_driver"]
    supplier = row["supplier_name"]
    
    if risk_level in ["Critical Supplier Risk", "High Supplier Risk"]:
        return (
            f"Prioritize audit review for {supplier}. Main risk driver: {driver}. "
            "Perform detailed invoice testing, supplier master data review and payment control validation."
        )
    elif risk_level == "Medium Supplier Risk":
        return (
            f"Perform targeted review for {supplier}. Main risk driver: {driver}. "
            "Focus on exception patterns and sample-based transaction testing."
        )
    else:
        return (
            f"Maintain standard monitoring for {supplier}. Main observed driver: {driver}."
        )

supplier_risk_summary["Supplier_Audit_Recommendation"] = supplier_risk_summary.apply(
    generate_supplier_recommendation,
    axis=1
)

# ------------------------------------------------------------
# Final supplier risk ranking
# ------------------------------------------------------------

supplier_risk_summary = supplier_risk_summary.sort_values(
    "supplier_risk_score",
    ascending=False
).reset_index(drop=True)

# Round selected columns
round_cols = [
    "Total_Invoice_Amount",
    "Average_Invoice_Amount",
    "Average_Audit_Risk_Score",
    "Max_Audit_Risk_Score",
    "Average_Rule_Count",
    "Spend_Share",
    "Exception_Rate",
    "High_Risk_Invoice_Rate",
    "Medium_Risk_Invoice_Rate",
    "Procurement_Exception_Rate",
    "Payment_Exception_Rate"
]

supplier_risk_summary[round_cols] = supplier_risk_summary[round_cols].round(2)

# ------------------------------------------------------------
# Supplier risk level distribution
# ------------------------------------------------------------

supplier_risk_level_distribution = (
    supplier_risk_summary["supplier_risk_level"]
    .value_counts()
    .reset_index()
)

supplier_risk_level_distribution.columns = ["Supplier_Risk_Level", "Suppliers"]

supplier_risk_level_distribution["Percentage"] = (
    supplier_risk_level_distribution["Suppliers"] /
    len(supplier_risk_summary) * 100
).round(2)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

supplier_risk_summary_path = REPORTS_DIR / "supplier_risk_summary.csv"
supplier_risk_distribution_path = REPORTS_DIR / "supplier_risk_level_distribution.csv"

supplier_risk_summary.to_csv(supplier_risk_summary_path, index=False)
supplier_risk_level_distribution.to_csv(supplier_risk_distribution_path, index=False)

# ------------------------------------------------------------
# Display outputs
# ------------------------------------------------------------

print("Supplier risk scoring completed successfully")
print("Supplier risk summary shape:", supplier_risk_summary.shape)

print("\nSupplier risk level distribution:")
display(supplier_risk_level_distribution)

print("\nTop 20 high-risk suppliers:")
display(
    supplier_risk_summary[
        [
            "supplier_id",
            "supplier_name",
            "supplier_category",
            "supplier_country",
            "Invoice_Count",
            "Total_Invoice_Amount",
            "Spend_Share",
            "Exception_Rate",
            "High_Risk_Invoice_Rate",
            "Critical_Rule_Triggers",
            "Procurement_Exception_Rate",
            "Payment_Exception_Rate",
            "supplier_risk_score",
            "supplier_risk_level",
            "primary_supplier_risk_driver"
        ]
    ].head(20)
)

print("\nSupplier recommendations sample:")
display(
    supplier_risk_summary[
        [
            "supplier_name",
            "supplier_risk_score",
            "supplier_risk_level",
            "primary_supplier_risk_driver",
            "Supplier_Audit_Recommendation"
        ]
    ].head(10)
)

print("\nFiles saved successfully:")
print("-", supplier_risk_summary_path)
print("-", supplier_risk_distribution_path)

SUPPLIER RISK SCORING
Supplier risk scoring completed successfully
Supplier risk summary shape: (320, 45)

Supplier risk level distribution:


,Supplier_Risk_Level,Suppliers,Percentage
0,Low Supplier Risk,168,52.5000
1,Medium Supplier Risk,139,43.4400
2,High Supplier Risk,13,4.0600



Top 20 high-risk suppliers:


,supplier_id,supplier_name,supplier_category,supplier_country,Invoice_Count,Total_Invoice_Amount,Spend_Share,Exception_Rate,High_Risk_Invoice_Rate,Critical_Rule_Triggers,Procurement_Exception_Rate,Payment_Exception_Rate,supplier_risk_score,supplier_risk_level,primary_supplier_risk_driver
0,SUP00184,Supplier_0184,Facilities,Algeria,8,"197,372.2300",0.1800,100.0000,12.5000,2,125.0000,87.5000,66.5900,High Supplier Risk,High exception rate
1,SUP00301,Supplier_0301,Facilities,France,17,"123,829.9300",0.1100,100.0000,5.8800,4,70.5900,64.7100,58.4200,High Supplier Risk,High exception rate
2,SUP00021,Supplier_0021,Maintenance,Algeria,119,"1,462,405.7100",1.3100,100.0000,1.6800,16,70.5900,47.9000,58.2100,High Supplier Risk,High exception rate
3,SUP00017,Supplier_0017,Logistics,Algeria,79,"1,049,765.6800",0.9400,100.0000,1.2700,9,78.4800,56.9600,55.6400,High Supplier Risk,High exception rate
4,SUP00147,Supplier_0147,Marketing Services,United Arab Emirates,17,"177,322.6200",0.1600,100.0000,5.8800,2,82.3500,47.0600,54.3300,High Supplier Risk,High exception rate
5,SUP00216,Supplier_0216,Marketing Services,Spain,7,"77,439.1900",0.0700,100.0000,0.0000,2,128.5700,128.5700,53.8600,High Supplier Risk,High exception rate
6,SUP00281,Supplier_0281,Telecom,Germany,8,"300,452.5100",0.2700,100.0000,0.0000,2,150.0000,62.5000,53.7600,High Supplier Risk,High average transaction risk
7,SUP00106,Supplier_0106,Office Supplies,Algeria,3,"37,655.5300",0.0300,100.0000,0.0000,1,166.6700,66.6700,53.1700,High Supplier Risk,High exception rate
8,SUP00009,Supplier_0009,IT Services,Algeria,325,"4,126,869.9800",3.7100,85.2300,0.0000,34,69.5400,52.3100,53.0200,High Supplier Risk,Critical audit rule triggers
9,SUP00030,Supplier_0030,Logistics,France,102,"1,276,173.3400",1.1500,100.0000,0.0000,9,65.6900,52.9400,51.9300,High Supplier Risk,High exception rate



Supplier recommendations sample:


,supplier_name,supplier_risk_score,supplier_risk_level,primary_supplier_risk_driver,Supplier_Audit_Recommendation
0,Supplier_0184,66.5900,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0184. Mai...
1,Supplier_0301,58.4200,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0301. Mai...
2,Supplier_0021,58.2100,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0021. Mai...
3,Supplier_0017,55.6400,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0017. Mai...
4,Supplier_0147,54.3300,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0147. Mai...
5,Supplier_0216,53.8600,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0216. Mai...
6,Supplier_0281,53.7600,High Supplier Risk,High average transaction risk,Prioritize audit review for Supplier_0281. Mai...
7,Supplier_0106,53.1700,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0106. Mai...
8,Supplier_0009,53.0200,High Supplier Risk,Critical audit rule triggers,Prioritize audit review for Supplier_0009. Mai...
9,Supplier_0030,51.9300,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0030. Mai...



Files saved successfully:
- ..\reports\supplier_risk_summary.csv
- ..\reports\supplier_risk_level_distribution.csv


# 7. Financial KPI Analysis & Audit Dashboard Metrics

This section creates executive-level audit and financial risk KPIs.

The objective is to summarize the audit findings into business metrics that can be used by audit teams, finance teams and risk advisory consultants.

The analysis covers:

- total invoice exposure;
- payment status;
- audit exception rate;
- risk level distribution;
- high-risk financial exposure;
- procurement control exposure;
- payment control exposure;
- department risk profile;
- supplier category risk profile;
- monthly audit risk trend.

These indicators will be used later in the automated Excel report and interactive audit dashboard.

In [13]:
# ============================================================
# 7. FINANCIAL KPI ANALYSIS & AUDIT DASHBOARD METRICS
# ============================================================

print("=" * 80)
print("FINANCIAL KPI ANALYSIS & AUDIT DASHBOARD METRICS")
print("=" * 80)

# ------------------------------------------------------------
# Executive audit KPIs
# ------------------------------------------------------------

total_invoices = len(audit_scored_df)
total_invoice_amount = audit_scored_df["invoice_amount"].sum()

paid_invoices = audit_scored_df["paid_flag"].sum()
unpaid_invoices = total_invoices - paid_invoices

exception_invoices = (audit_scored_df["audit_rule_count"] > 0).sum()
exception_rate = exception_invoices / total_invoices * 100

no_exception_invoices = (audit_scored_df["audit_risk_level"] == "No Exception").sum()
low_risk_invoices = (audit_scored_df["audit_risk_level"] == "Low Risk").sum()
medium_risk_invoices = (audit_scored_df["audit_risk_level"] == "Medium Risk").sum()
high_risk_invoices = audit_scored_df["audit_risk_level"].isin(["High Risk", "Critical Risk"]).sum()

high_risk_amount = audit_scored_df.loc[
    audit_scored_df["audit_risk_level"].isin(["High Risk", "Critical Risk"]),
    "invoice_amount"
].sum()

medium_high_risk_amount = audit_scored_df.loc[
    audit_scored_df["audit_risk_level"].isin(["Medium Risk", "High Risk", "Critical Risk"]),
    "invoice_amount"
].sum()

average_risk_score = audit_scored_df["audit_risk_score"].mean()
median_risk_score = audit_scored_df["audit_risk_score"].median()

duplicate_risk_invoices = audit_scored_df["audit_duplicate_risk_flag"].sum()
missing_po_invoices = audit_scored_df["missing_po_flag"].sum()
late_payment_invoices = audit_scored_df["late_payment_flag"].sum()
very_late_payment_invoices = audit_scored_df["very_late_payment_flag"].sum()
payment_before_invoice_invoices = audit_scored_df["payment_before_invoice_flag"].sum()

procurement_exception_invoices = (
    audit_scored_df[
        [
            "missing_po_flag",
            "po_supplier_mismatch_flag",
            "po_department_mismatch_flag",
            "invoice_exceeds_po_amount_flag",
            "po_not_approved_flag"
        ]
    ].sum(axis=1) > 0
).sum()

payment_exception_invoices = (
    audit_scored_df[
        [
            "payment_before_invoice_flag",
            "very_late_payment_flag",
            "payment_amount_mismatch_flag",
            "weekend_payment_flag",
            "cash_payment_flag",
            "pending_or_failed_payment_flag"
        ]
    ].sum(axis=1) > 0
).sum()

high_supplier_risk_count = supplier_risk_summary[
    supplier_risk_summary["supplier_risk_level"] == "High Supplier Risk"
].shape[0]

top_risk_supplier = supplier_risk_summary.iloc[0]["supplier_name"]
top_risk_supplier_score = supplier_risk_summary.iloc[0]["supplier_risk_score"]

# ------------------------------------------------------------
# Executive KPI table
# ------------------------------------------------------------

executive_audit_kpis = pd.DataFrame({
    "KPI": [
        "Total Invoices",
        "Total Invoice Amount",
        "Paid Invoices",
        "Unpaid Invoices",
        "Paid Invoice Share (%)",
        "Invoices with Audit Exceptions",
        "Audit Exception Rate (%)",
        "No Exception Invoices",
        "Low Risk Invoices",
        "Medium Risk Invoices",
        "High/Critical Risk Invoices",
        "High/Critical Risk Exposure",
        "Medium-to-High Risk Exposure",
        "Average Audit Risk Score",
        "Median Audit Risk Score",
        "Duplicate Risk Invoices",
        "Missing PO Invoices",
        "Late Payment Invoices",
        "Very Late Payment Invoices",
        "Payment Before Invoice Date",
        "Procurement Exception Invoices",
        "Payment Exception Invoices",
        "High-Risk Suppliers",
        "Top Risk Supplier",
        "Top Risk Supplier Score"
    ],
    "Value": [
        total_invoices,
        round(total_invoice_amount, 2),
        int(paid_invoices),
        int(unpaid_invoices),
        round(paid_invoices / total_invoices * 100, 2),
        int(exception_invoices),
        round(exception_rate, 2),
        int(no_exception_invoices),
        int(low_risk_invoices),
        int(medium_risk_invoices),
        int(high_risk_invoices),
        round(high_risk_amount, 2),
        round(medium_high_risk_amount, 2),
        round(average_risk_score, 2),
        round(median_risk_score, 2),
        int(duplicate_risk_invoices),
        int(missing_po_invoices),
        int(late_payment_invoices),
        int(very_late_payment_invoices),
        int(payment_before_invoice_invoices),
        int(procurement_exception_invoices),
        int(payment_exception_invoices),
        int(high_supplier_risk_count),
        top_risk_supplier,
        round(top_risk_supplier_score, 2)
    ]
})

print("Executive audit KPIs:")
display(executive_audit_kpis)

# ------------------------------------------------------------
# Department-level risk summary
# ------------------------------------------------------------

department_risk_summary = (
    audit_scored_df
    .groupby(["department_id", "department_name", "business_unit"])
    .agg(
        Invoice_Count=("invoice_id", "count"),
        Total_Invoice_Amount=("invoice_amount", "sum"),
        Average_Invoice_Amount=("invoice_amount", "mean"),
        Exception_Invoices=("audit_rule_count", lambda x: (x > 0).sum()),
        Average_Risk_Score=("audit_risk_score", "mean"),
        High_Risk_Invoices=("audit_risk_level", lambda x: x.isin(["High Risk", "Critical Risk"]).sum()),
        Medium_Risk_Invoices=("audit_risk_level", lambda x: (x == "Medium Risk").sum()),
        Missing_PO_Count=("missing_po_flag", "sum"),
        Duplicate_Risk_Count=("audit_duplicate_risk_flag", "sum"),
        Late_Payment_Count=("late_payment_flag", "sum"),
        Very_Late_Payment_Count=("very_late_payment_flag", "sum"),
        PO_Not_Approved_Count=("po_not_approved_flag", "sum"),
        Above_Threshold_Count=("above_department_threshold_flag", "sum")
    )
    .reset_index()
)

department_risk_summary["Exception_Rate"] = (
    department_risk_summary["Exception_Invoices"] /
    department_risk_summary["Invoice_Count"] * 100
)

department_risk_summary["High_Risk_Rate"] = (
    department_risk_summary["High_Risk_Invoices"] /
    department_risk_summary["Invoice_Count"] * 100
)

department_risk_summary["Spend_Share"] = (
    department_risk_summary["Total_Invoice_Amount"] /
    department_risk_summary["Total_Invoice_Amount"].sum() * 100
)

round_cols = [
    "Total_Invoice_Amount",
    "Average_Invoice_Amount",
    "Average_Risk_Score",
    "Exception_Rate",
    "High_Risk_Rate",
    "Spend_Share"
]

department_risk_summary[round_cols] = department_risk_summary[round_cols].round(2)

department_risk_summary = department_risk_summary.sort_values(
    ["Average_Risk_Score", "Total_Invoice_Amount"],
    ascending=False
)

print("\nDepartment risk summary:")
display(department_risk_summary)

# ------------------------------------------------------------
# Business unit risk summary
# ------------------------------------------------------------

business_unit_risk_summary = (
    audit_scored_df
    .groupby("business_unit")
    .agg(
        Invoice_Count=("invoice_id", "count"),
        Total_Invoice_Amount=("invoice_amount", "sum"),
        Exception_Invoices=("audit_rule_count", lambda x: (x > 0).sum()),
        Average_Risk_Score=("audit_risk_score", "mean"),
        High_Risk_Invoices=("audit_risk_level", lambda x: x.isin(["High Risk", "Critical Risk"]).sum()),
        Medium_Risk_Invoices=("audit_risk_level", lambda x: (x == "Medium Risk").sum())
    )
    .reset_index()
)

business_unit_risk_summary["Exception_Rate"] = (
    business_unit_risk_summary["Exception_Invoices"] /
    business_unit_risk_summary["Invoice_Count"] * 100
)

business_unit_risk_summary["High_Risk_Rate"] = (
    business_unit_risk_summary["High_Risk_Invoices"] /
    business_unit_risk_summary["Invoice_Count"] * 100
)

business_unit_risk_summary[
    [
        "Total_Invoice_Amount",
        "Average_Risk_Score",
        "Exception_Rate",
        "High_Risk_Rate"
    ]
] = business_unit_risk_summary[
    [
        "Total_Invoice_Amount",
        "Average_Risk_Score",
        "Exception_Rate",
        "High_Risk_Rate"
    ]
].round(2)

business_unit_risk_summary = business_unit_risk_summary.sort_values(
    "Average_Risk_Score",
    ascending=False
)

print("\nBusiness unit risk summary:")
display(business_unit_risk_summary)

# ------------------------------------------------------------
# Monthly audit risk trend
# ------------------------------------------------------------

monthly_audit_trend = (
    audit_scored_df
    .groupby(["invoice_year", "invoice_month"])
    .agg(
        Invoice_Count=("invoice_id", "count"),
        Total_Invoice_Amount=("invoice_amount", "sum"),
        Exception_Invoices=("audit_rule_count", lambda x: (x > 0).sum()),
        Average_Risk_Score=("audit_risk_score", "mean"),
        High_Risk_Invoices=("audit_risk_level", lambda x: x.isin(["High Risk", "Critical Risk"]).sum()),
        Duplicate_Risk_Count=("audit_duplicate_risk_flag", "sum"),
        Missing_PO_Count=("missing_po_flag", "sum"),
        Late_Payment_Count=("late_payment_flag", "sum")
    )
    .reset_index()
)

monthly_audit_trend["Month"] = pd.to_datetime(
    monthly_audit_trend["invoice_year"].astype(str) + "-" +
    monthly_audit_trend["invoice_month"].astype(str) + "-01"
)

monthly_audit_trend["Exception_Rate"] = (
    monthly_audit_trend["Exception_Invoices"] /
    monthly_audit_trend["Invoice_Count"] * 100
)

monthly_audit_trend[
    [
        "Total_Invoice_Amount",
        "Average_Risk_Score",
        "Exception_Rate"
    ]
] = monthly_audit_trend[
    [
        "Total_Invoice_Amount",
        "Average_Risk_Score",
        "Exception_Rate"
    ]
].round(2)

monthly_audit_trend = monthly_audit_trend.sort_values("Month")

print("\nMonthly audit risk trend:")
display(monthly_audit_trend)

# ------------------------------------------------------------
# Supplier category risk summary
# ------------------------------------------------------------

supplier_category_risk_summary = (
    audit_scored_df
    .groupby("supplier_category")
    .agg(
        Invoice_Count=("invoice_id", "count"),
        Total_Invoice_Amount=("invoice_amount", "sum"),
        Exception_Invoices=("audit_rule_count", lambda x: (x > 0).sum()),
        Average_Risk_Score=("audit_risk_score", "mean"),
        High_Risk_Invoices=("audit_risk_level", lambda x: x.isin(["High Risk", "Critical Risk"]).sum()),
        Duplicate_Risk_Count=("audit_duplicate_risk_flag", "sum"),
        Missing_Tax_ID_Count=("missing_tax_id_flag", "sum")
    )
    .reset_index()
)

supplier_category_risk_summary["Exception_Rate"] = (
    supplier_category_risk_summary["Exception_Invoices"] /
    supplier_category_risk_summary["Invoice_Count"] * 100
)

supplier_category_risk_summary["Spend_Share"] = (
    supplier_category_risk_summary["Total_Invoice_Amount"] /
    supplier_category_risk_summary["Total_Invoice_Amount"].sum() * 100
)

supplier_category_risk_summary[
    [
        "Total_Invoice_Amount",
        "Average_Risk_Score",
        "Exception_Rate",
        "Spend_Share"
    ]
] = supplier_category_risk_summary[
    [
        "Total_Invoice_Amount",
        "Average_Risk_Score",
        "Exception_Rate",
        "Spend_Share"
    ]
].round(2)

supplier_category_risk_summary = supplier_category_risk_summary.sort_values(
    "Average_Risk_Score",
    ascending=False
)

print("\nSupplier category risk summary:")
display(supplier_category_risk_summary)

# ------------------------------------------------------------
# Currency exposure summary
# ------------------------------------------------------------

currency_exposure_summary = (
    audit_scored_df
    .groupby("currency")
    .agg(
        Invoice_Count=("invoice_id", "count"),
        Total_Invoice_Amount=("invoice_amount", "sum"),
        Average_Risk_Score=("audit_risk_score", "mean"),
        Exception_Invoices=("audit_rule_count", lambda x: (x > 0).sum())
    )
    .reset_index()
)

currency_exposure_summary["Exception_Rate"] = (
    currency_exposure_summary["Exception_Invoices"] /
    currency_exposure_summary["Invoice_Count"] * 100
)

currency_exposure_summary[
    [
        "Total_Invoice_Amount",
        "Average_Risk_Score",
        "Exception_Rate"
    ]
] = currency_exposure_summary[
    [
        "Total_Invoice_Amount",
        "Average_Risk_Score",
        "Exception_Rate"
    ]
].round(2)

currency_exposure_summary = currency_exposure_summary.sort_values(
    "Total_Invoice_Amount",
    ascending=False
)

print("\nCurrency exposure summary:")
display(currency_exposure_summary)

# ------------------------------------------------------------
# Save KPI outputs
# ------------------------------------------------------------

executive_kpis_path = REPORTS_DIR / "executive_audit_kpis.csv"
department_risk_path = REPORTS_DIR / "department_risk_summary.csv"
business_unit_risk_path = REPORTS_DIR / "business_unit_risk_summary.csv"
monthly_trend_path = REPORTS_DIR / "monthly_audit_risk_trend.csv"
supplier_category_risk_path = REPORTS_DIR / "supplier_category_risk_summary.csv"
currency_exposure_path = REPORTS_DIR / "currency_exposure_summary.csv"

executive_audit_kpis.to_csv(executive_kpis_path, index=False)
department_risk_summary.to_csv(department_risk_path, index=False)
business_unit_risk_summary.to_csv(business_unit_risk_path, index=False)
monthly_audit_trend.to_csv(monthly_trend_path, index=False)
supplier_category_risk_summary.to_csv(supplier_category_risk_path, index=False)
currency_exposure_summary.to_csv(currency_exposure_path, index=False)

print("\nFinancial KPI reports saved successfully:")
print("-", executive_kpis_path)
print("-", department_risk_path)
print("-", business_unit_risk_path)
print("-", monthly_trend_path)
print("-", supplier_category_risk_path)
print("-", currency_exposure_path)

FINANCIAL KPI ANALYSIS & AUDIT DASHBOARD METRICS
Executive audit KPIs:


,KPI,Value
0,Total Invoices,8630
1,Total Invoice Amount,"111,360,795.8300"
2,Paid Invoices,7918
3,Unpaid Invoices,712
4,Paid Invoice Share (%),91.7500
5,Invoices with Audit Exceptions,7116
6,Audit Exception Rate (%),82.4600
7,No Exception Invoices,1514
8,Low Risk Invoices,5970
9,Medium Risk Invoices,1125



Department risk summary:


,department_id,department_name,business_unit,Invoice_Count,Total_Invoice_Amount,Average_Invoice_Amount,Exception_Invoices,Average_Risk_Score,High_Risk_Invoices,Medium_Risk_Invoices,Missing_PO_Count,Duplicate_Risk_Count,Late_Payment_Count,Very_Late_Payment_Count,PO_Not_Approved_Count,Above_Threshold_Count,Exception_Rate,High_Risk_Rate,Spend_Share
11,DPT012,General Administration,Support,385,"4,806,487.8400","12,484.3800",331,9.0100,0,79,74,5,224,62,29,233,85.9700,0.0000,4.3200
2,DPT003,IT,Operations,945,"12,074,008.0600","12,776.7300",823,8.8800,3,159,176,30,546,146,79,532,87.0900,0.3200,10.8400
8,DPT009,Risk Management,Commercial,718,"9,080,001.1800","12,646.2400",612,8.5800,2,114,133,24,411,93,79,250,85.2400,0.2800,8.1500
3,DPT004,Operations,Technology,1043,"14,305,206.9700","13,715.4400",883,8.4600,5,160,168,35,598,144,98,388,84.6600,0.4800,12.8500
4,DPT005,Sales,Technology,702,"8,214,616.0600","11,701.7300",573,8.0900,1,100,123,18,401,105,75,238,81.6200,0.1400,7.3800
9,DPT010,Customer Support,Corporate,402,"5,593,523.1300","13,914.2400",322,7.9200,0,58,63,17,226,48,35,148,80.1000,0.0000,5.0200
7,DPT008,Legal,Operations,399,"5,717,785.4000","14,330.2900",324,7.7200,1,51,77,18,224,56,31,21,81.2000,0.2500,5.1300
10,DPT011,Logistics,Technology,483,"6,110,813.4900","12,651.7900",388,7.7100,4,55,80,13,293,73,59,57,80.3300,0.8300,5.4900
6,DPT007,Human Resources,Corporate,562,"7,693,982.7700","13,690.3600",458,7.5800,1,61,95,13,318,80,51,66,81.4900,0.1800,6.9100
0,DPT001,Finance,Corporate,1163,"14,580,303.8300","12,536.8000",945,7.2500,1,112,207,43,647,167,106,39,81.2600,0.0900,13.0900



Business unit risk summary:


...


Supplier category risk summary:


,supplier_category,Invoice_Count,Total_Invoice_Amount,Exception_Invoices,Average_Risk_Score,High_Risk_Invoices,Duplicate_Risk_Count,Missing_Tax_ID_Count,Exception_Rate,Spend_Share
4,Maintenance,483,"6,789,428.3200",409,9.1400,2,21,133,84.6800,6.1000
3,Logistics,895,"11,352,132.2100",758,8.6600,2,26,181,84.6900,10.1900
9,Training,878,"10,535,762.6000",733,8.0700,4,29,30,83.4900,9.4600
1,Facilities,891,"11,981,678.1100",709,7.8500,3,16,48,79.5700,10.7600
7,Professional Services,930,"12,154,454.8200",763,7.8200,3,36,35,82.0400,10.9100
8,Telecom,1377,"19,214,480.8700",1133,7.7900,4,47,11,82.2800,17.2500
5,Marketing Services,900,"11,848,044.2500",743,7.7600,2,23,43,82.5600,10.6400
6,Office Supplies,654,"7,868,948.9600",536,7.6000,1,20,11,81.9600,7.0700
2,IT Services,783,"9,343,055.1300",640,7.4700,0,22,0,81.7400,8.3900
0,Consulting,839,"10,272,810.5600",692,7.2700,0,25,10,82.4800,9.2200



Currency exposure summary:


,currency,Invoice_Count,Total_Invoice_Amount,Average_Risk_Score,Exception_Invoices,Exception_Rate
0,DZD,7082,"91,189,049.8500",7.8600,5819,82.1700
1,EUR,982,"12,403,268.8100",8.0800,826,84.1100
2,USD,566,"7,768,477.1700",8.0400,471,83.2200



Financial KPI reports saved successfully:
- ..\reports\executive_audit_kpis.csv
- ..\reports\department_risk_summary.csv
- ..\reports\business_unit_risk_summary.csv
- ..\reports\monthly_audit_risk_trend.csv
- ..\reports\supplier_category_risk_summary.csv
- ..\reports\currency_exposure_summary.csv


# 8. Machine Learning Anomaly Detection

This section applies an unsupervised machine learning model to identify unusual financial transactions.

The objective is to complement the audit rules engine with anomaly detection.

The model used is Isolation Forest, which is suitable for detecting unusual patterns when no labeled fraud variable is available.

The anomaly detection model uses financial, payment, procurement and categorical features to identify invoices that behave differently from the rest of the population.

The final output includes:

- anomaly flag;
- anomaly score;
- combined review priority score;
- top anomalous transactions;
- anomaly distribution by department and supplier;
- comparison between audit risk levels and ML anomaly detection.

In [14]:
# ============================================================
# 8. MACHINE LEARNING ANOMALY DETECTION
# ============================================================

print("=" * 80)
print("MACHINE LEARNING ANOMALY DETECTION")
print("=" * 80)

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# Create modeling dataset
# ------------------------------------------------------------

ml_df = audit_scored_df.copy()

# ------------------------------------------------------------
# Feature selection
# ------------------------------------------------------------

numeric_features = [
    "invoice_amount",
    "payment_terms_days",
    "invoice_due_days",
    "paid_flag",
    "payment_count",
    "total_paid_amount",
    "days_to_payment",
    "days_late",
    "payment_amount_difference_pct",
    "po_amount",
    "po_amount_difference_pct",
    "approval_threshold",
    "invoice_month",
    "audit_rule_count",
    "audit_risk_score"
]

categorical_features = [
    "currency",
    "invoice_status",
    "supplier_category",
    "supplier_country",
    "department_name",
    "business_unit",
    "audit_payment_status"
]

# Keep only existing columns
numeric_features = [col for col in numeric_features if col in ml_df.columns]
categorical_features = [col for col in categorical_features if col in ml_df.columns]

ml_features = ml_df[numeric_features + categorical_features].copy()

# ------------------------------------------------------------
# Clean numerical features
# ------------------------------------------------------------

for col in numeric_features:
    ml_features[col] = pd.to_numeric(ml_features[col], errors="coerce")
    ml_features[col] = ml_features[col].replace([np.inf, -np.inf], np.nan)
    ml_features[col] = ml_features[col].fillna(ml_features[col].median())

# ------------------------------------------------------------
# Clean categorical features
# ------------------------------------------------------------

for col in categorical_features:
    ml_features[col] = ml_features[col].fillna("Unknown").astype(str)

# ------------------------------------------------------------
# Add log-transformed financial variables
# ------------------------------------------------------------

log_transform_cols = [
    "invoice_amount",
    "total_paid_amount",
    "po_amount",
    "approval_threshold"
]

for col in log_transform_cols:
    if col in ml_features.columns:
        ml_features[f"log_{col}"] = np.log1p(ml_features[col].clip(lower=0))

# ------------------------------------------------------------
# One-hot encoding
# ------------------------------------------------------------

ml_features_encoded = pd.get_dummies(
    ml_features,
    columns=categorical_features,
    drop_first=False
)

# ------------------------------------------------------------
# Scale features
# ------------------------------------------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(ml_features_encoded)

# ------------------------------------------------------------
# Isolation Forest model
# ------------------------------------------------------------

isolation_model = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

isolation_model.fit(X_scaled)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

ml_df["ml_anomaly_prediction"] = isolation_model.predict(X_scaled)
ml_df["ml_anomaly_flag"] = np.where(
    ml_df["ml_anomaly_prediction"] == -1,
    1,
    0
)

# Higher score = more anomalous
raw_anomaly_score = -isolation_model.decision_function(X_scaled)

ml_df["ml_anomaly_score"] = (
    (raw_anomaly_score - raw_anomaly_score.min()) /
    (raw_anomaly_score.max() - raw_anomaly_score.min()) * 100
).round(2)

# Combined score: audit rules + ML anomaly
ml_df["combined_review_priority_score"] = (
    0.60 * ml_df["ml_anomaly_score"] +
    0.40 * ml_df["audit_risk_score"]
).round(2)

# ------------------------------------------------------------
# Priority classification
# ------------------------------------------------------------

def classify_ml_priority(score):
    if score >= 70:
        return "Critical Review Priority"
    elif score >= 50:
        return "High Review Priority"
    elif score >= 30:
        return "Medium Review Priority"
    elif score > 0:
        return "Low Review Priority"
    else:
        return "No Review Priority"

ml_df["ml_review_priority"] = ml_df["combined_review_priority_score"].apply(
    classify_ml_priority
)

print("Machine learning anomaly detection completed successfully")
print("ML dataset shape:", ml_df.shape)
print("Encoded feature matrix shape:", ml_features_encoded.shape)
print("Number of detected anomalies:", ml_df["ml_anomaly_flag"].sum())
print("Detected anomaly rate (%):", round(ml_df["ml_anomaly_flag"].mean() * 100, 2))

MACHINE LEARNING ANOMALY DETECTION
Machine learning anomaly detection completed successfully
ML dataset shape: (8630, 94)
Encoded feature matrix shape: (8630, 66)
Number of detected anomalies: 432
Detected anomaly rate (%): 5.01


In [15]:
# ============================================================
# 9. ANOMALY DETECTION RESULTS
# ============================================================

# ------------------------------------------------------------
# Anomaly distribution
# ------------------------------------------------------------

anomaly_distribution = (
    ml_df["ml_anomaly_flag"]
    .value_counts()
    .reset_index()
)

anomaly_distribution.columns = ["ML_Anomaly_Flag", "Invoices"]

anomaly_distribution["Label"] = anomaly_distribution["ML_Anomaly_Flag"].map({
    0: "Normal Transaction",
    1: "Detected Anomaly"
})

anomaly_distribution["Percentage"] = (
    anomaly_distribution["Invoices"] / len(ml_df) * 100
).round(2)

print("Anomaly distribution:")
display(anomaly_distribution)

# ------------------------------------------------------------
# Cross-tab: audit risk level vs ML anomaly
# ------------------------------------------------------------

audit_ml_crosstab = pd.crosstab(
    ml_df["audit_risk_level"],
    ml_df["ml_anomaly_flag"],
    margins=True
)

audit_ml_crosstab = audit_ml_crosstab.rename(columns={
    0: "Normal by ML",
    1: "Anomaly by ML"
})

print("\nAudit risk level vs ML anomaly detection:")
display(audit_ml_crosstab)

# ------------------------------------------------------------
# ML review priority distribution
# ------------------------------------------------------------

ml_priority_distribution = (
    ml_df["ml_review_priority"]
    .value_counts()
    .reset_index()
)

ml_priority_distribution.columns = ["ML_Review_Priority", "Invoices"]

ml_priority_distribution["Percentage"] = (
    ml_priority_distribution["Invoices"] / len(ml_df) * 100
).round(2)

print("\nML review priority distribution:")
display(ml_priority_distribution)

# ------------------------------------------------------------
# Top anomalous transactions
# ------------------------------------------------------------

top_anomalous_transactions = (
    ml_df
    .sort_values(
        ["combined_review_priority_score", "ml_anomaly_score", "audit_risk_score"],
        ascending=False
    )
    [
        [
            "invoice_id",
            "invoice_number",
            "supplier_name",
            "department_name",
            "supplier_category",
            "invoice_date",
            "invoice_amount",
            "paid_flag",
            "audit_rule_count",
            "audit_risk_score",
            "audit_risk_level",
            "ml_anomaly_flag",
            "ml_anomaly_score",
            "combined_review_priority_score",
            "ml_review_priority",
            "primary_risk_driver"
        ]
    ]
    .head(25)
)

print("\nTop 25 anomalous transactions:")
display(top_anomalous_transactions)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

ml_anomaly_results_path = PROCESSED_DATA_DIR / "ml_anomaly_detection_results.csv"
anomaly_distribution_path = REPORTS_DIR / "ml_anomaly_distribution.csv"
audit_ml_crosstab_path = REPORTS_DIR / "audit_ml_crosstab.csv"
ml_priority_distribution_path = REPORTS_DIR / "ml_review_priority_distribution.csv"
top_anomalies_path = REPORTS_DIR / "top_anomalous_transactions.csv"

ml_df.to_csv(ml_anomaly_results_path, index=False)
anomaly_distribution.to_csv(anomaly_distribution_path, index=False)
audit_ml_crosstab.to_csv(audit_ml_crosstab_path)
ml_priority_distribution.to_csv(ml_priority_distribution_path, index=False)
top_anomalous_transactions.to_csv(top_anomalies_path, index=False)

print("\nML anomaly outputs saved successfully:")
print("-", ml_anomaly_results_path)
print("-", anomaly_distribution_path)
print("-", audit_ml_crosstab_path)
print("-", ml_priority_distribution_path)
print("-", top_anomalies_path)

Anomaly distribution:


,ML_Anomaly_Flag,Invoices,Label,Percentage
0,0,8198,Normal Transaction,94.9900
1,1,432,Detected Anomaly,5.0100



Audit risk level vs ML anomaly detection:


ml_anomaly_flag,Normal by ML,Anomaly by ML,All
audit_risk_level,,,
High Risk,11,10,21
Low Risk,5747,223,5970
Medium Risk,994,131,1125
No Exception,1446,68,1514
All,8198,432,8630



ML review priority distribution:


,ML_Review_Priority,Invoices,Percentage
0,Low Review Priority,6672,77.3100
1,Medium Review Priority,1850,21.4400
2,High Review Priority,107,1.2400
3,Critical Review Priority,1,0.0100



Top 25 anomalous transactions:


,invoice_id,invoice_number,supplier_name,department_name,supplier_category,invoice_date,invoice_amount,paid_flag,audit_rule_count,audit_risk_score,audit_risk_level,ml_anomaly_flag,ml_anomaly_score,combined_review_priority_score,ml_review_priority,primary_risk_driver
6304,INV0006305,FCT-360582,Supplier_0101,Logistics,Professional Services,2024-07-08,"100,000.0000",1,6,30.1100,High Risk,1,98.5800,71.1900,Critical Review Priority,R13 - PO department mismatch
2034,INV0002035,FCT-528008,Supplier_0002,Risk Management,Training,2024-09-18,"100,000.0000",1,6,30.1100,High Risk,1,94.4500,68.7100,High Review Priority,R15 - PO not approved
1100,INV0001101,FCT-253802,Supplier_0018,Operations,Marketing Services,2024-04-05,"106,166.2600",0,4,21.5100,Medium Risk,1,99.8800,68.5300,High Review Priority,R15 - PO not approved
6515,INV0006516,FCT-701982,Supplier_0074,Marketing,Telecom,2024-06-19,"138,721.0300",1,6,33.8700,High Risk,1,88.0000,66.3500,High Review Priority,R06 - Payment before invoice date
6587,INV0006588,FCT-546673,Supplier_0016,IT,Consulting,2024-12-27,"207,315.9900",1,4,17.7400,Medium Risk,1,96.8900,65.2300,High Review Priority,R04 - High-value invoice
8589,INV0008590,FCT-533287,Supplier_0123,Legal,Maintenance,2024-11-24,706.9100,0,2,16.1300,Medium Risk,1,97.2600,64.8100,High Review Priority,R02 - Potential duplicate invoice
4162,INV0004163,FCT-759856,Supplier_0003,Marketing,Office Supplies,2024-07-17,"96,009.3000",1,4,18.2800,Medium Risk,1,95.7000,64.7300,High Review Priority,R13 - PO department mismatch
7813,INV0007814,FCT-396205,Supplier_0006,Procurement,Telecom,2024-12-09,"100,000.0000",0,5,27.9600,Medium Risk,1,88.4300,64.2400,High Review Priority,R02 - Potential duplicate invoice
2335,INV0002336,FCT-891175,Supplier_0003,General Administration,Office Supplies,2024-09-17,"41,780.4400",0,2,9.6800,Low Risk,1,100.0000,63.8700,High Review Priority,R01 - Invoice without purchase order
1589,INV0001590,FCT-458849,Supplier_0281,Marketing,Telecom,2024-12-17,"66,438.9100",0,3,22.5800,Medium Risk,1,90.8800,63.5600,High Review Priority,R12 - PO supplier mismatch



ML anomaly outputs saved successfully:
- ..\data\processed\ml_anomaly_detection_results.csv
- ..\reports\ml_anomaly_distribution.csv
- ..\reports\audit_ml_crosstab.csv
- ..\reports\ml_review_priority_distribution.csv
- ..\reports\top_anomalous_transactions.csv


In [16]:
# ============================================================
# 10. ANOMALY SUMMARY BY SUPPLIER AND DEPARTMENT
# ============================================================

# ------------------------------------------------------------
# Supplier anomaly summary
# ------------------------------------------------------------

supplier_anomaly_summary = (
    ml_df
    .groupby(["supplier_id", "supplier_name", "supplier_category", "supplier_country"])
    .agg(
        Invoice_Count=("invoice_id", "count"),
        Total_Invoice_Amount=("invoice_amount", "sum"),
        Anomaly_Count=("ml_anomaly_flag", "sum"),
        Average_Anomaly_Score=("ml_anomaly_score", "mean"),
        Average_Audit_Risk_Score=("audit_risk_score", "mean"),
        Max_Combined_Priority_Score=("combined_review_priority_score", "max")
    )
    .reset_index()
)

supplier_anomaly_summary["Anomaly_Rate"] = (
    supplier_anomaly_summary["Anomaly_Count"] /
    supplier_anomaly_summary["Invoice_Count"] * 100
)

supplier_anomaly_summary[
    [
        "Total_Invoice_Amount",
        "Average_Anomaly_Score",
        "Average_Audit_Risk_Score",
        "Max_Combined_Priority_Score",
        "Anomaly_Rate"
    ]
] = supplier_anomaly_summary[
    [
        "Total_Invoice_Amount",
        "Average_Anomaly_Score",
        "Average_Audit_Risk_Score",
        "Max_Combined_Priority_Score",
        "Anomaly_Rate"
    ]
].round(2)

supplier_anomaly_summary = supplier_anomaly_summary.sort_values(
    ["Anomaly_Count", "Max_Combined_Priority_Score"],
    ascending=False
)

print("Top suppliers by anomaly count:")
display(supplier_anomaly_summary.head(20))

# ------------------------------------------------------------
# Department anomaly summary
# ------------------------------------------------------------

department_anomaly_summary = (
    ml_df
    .groupby(["department_id", "department_name", "business_unit"])
    .agg(
        Invoice_Count=("invoice_id", "count"),
        Total_Invoice_Amount=("invoice_amount", "sum"),
        Anomaly_Count=("ml_anomaly_flag", "sum"),
        Average_Anomaly_Score=("ml_anomaly_score", "mean"),
        Average_Audit_Risk_Score=("audit_risk_score", "mean"),
        Max_Combined_Priority_Score=("combined_review_priority_score", "max")
    )
    .reset_index()
)

department_anomaly_summary["Anomaly_Rate"] = (
    department_anomaly_summary["Anomaly_Count"] /
    department_anomaly_summary["Invoice_Count"] * 100
)

department_anomaly_summary[
    [
        "Total_Invoice_Amount",
        "Average_Anomaly_Score",
        "Average_Audit_Risk_Score",
        "Max_Combined_Priority_Score",
        "Anomaly_Rate"
    ]
] = department_anomaly_summary[
    [
        "Total_Invoice_Amount",
        "Average_Anomaly_Score",
        "Average_Audit_Risk_Score",
        "Max_Combined_Priority_Score",
        "Anomaly_Rate"
    ]
].round(2)

department_anomaly_summary = department_anomaly_summary.sort_values(
    "Anomaly_Count",
    ascending=False
)

print("\nDepartment anomaly summary:")
display(department_anomaly_summary)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

supplier_anomaly_summary_path = REPORTS_DIR / "supplier_anomaly_summary.csv"
department_anomaly_summary_path = REPORTS_DIR / "department_anomaly_summary.csv"

supplier_anomaly_summary.to_csv(supplier_anomaly_summary_path, index=False)
department_anomaly_summary.to_csv(department_anomaly_summary_path, index=False)

print("\nAnomaly summary files saved successfully:")
print("-", supplier_anomaly_summary_path)
print("-", department_anomaly_summary_path)

Top suppliers by anomaly count:


,supplier_id,supplier_name,supplier_category,supplier_country,Invoice_Count,Total_Invoice_Amount,Anomaly_Count,Average_Anomaly_Score,Average_Audit_Risk_Score,Max_Combined_Priority_Score,Anomaly_Rate
2,SUP00003,Supplier_0003,Office Supplies,Turkey,282,"3,549,871.8500",30,39.5300,7.1900,64.7300,10.6400
1,SUP00002,Supplier_0002,Training,Germany,282,"3,877,256.3800",20,38.0300,8.2500,68.7100,7.0900
0,SUP00001,Supplier_0001,Training,Turkey,233,"2,431,526.3300",17,38.3000,7.9400,53.9200,7.3000
7,SUP00008,Supplier_0008,Logistics,France,269,"3,250,205.0300",14,34.2500,7.7600,59.4700,5.2000
3,SUP00004,Supplier_0004,Professional Services,Turkey,266,"3,516,104.7200",14,35.2600,7.3800,52.7400,5.2600
9,SUP00010,Supplier_0010,Facilities,Algeria,274,"3,646,952.1000",13,29.7500,7.5600,58.6000,4.7400
6,SUP00007,Supplier_0007,Telecom,Germany,264,"3,435,461.6100",12,32.9500,7.2000,59.0900,4.5500
22,SUP00023,Supplier_0023,Telecom,Germany,112,"1,468,913.0200",10,34.5100,7.5200,59.3300,8.9300
12,SUP00013,Supplier_0013,Professional Services,Tunisia,117,"1,345,019.7700",10,37.2200,7.6300,54.1900,8.5500
29,SUP00030,Supplier_0030,Logistics,France,102,"1,276,173.3400",9,35.3800,12.9100,60.2700,8.8200



Department anomaly summary:


,department_id,department_name,business_unit,Invoice_Count,Total_Invoice_Amount,Anomaly_Count,Average_Anomaly_Score,Average_Audit_Risk_Score,Max_Combined_Priority_Score,Anomaly_Rate
5,DPT006,Marketing,Support,671,"8,260,733.3500",78,43.2600,7.1600,66.3500,11.6200
3,DPT004,Operations,Technology,1043,"14,305,206.9700",53,30.9500,8.4600,68.5300,5.0800
8,DPT009,Risk Management,Commercial,718,"9,080,001.1800",53,38.1900,8.5800,68.7100,7.3800
2,DPT003,IT,Operations,945,"12,074,008.0600",46,31.2200,8.8800,65.2300,4.8700
1,DPT002,Procurement,Operations,1157,"14,923,333.7500",33,28.0000,7.0600,64.2400,2.8500
11,DPT012,General Administration,Support,385,"4,806,487.8400",28,40.0200,9.0100,63.8700,7.2700
6,DPT007,Human Resources,Corporate,562,"7,693,982.7700",26,31.9300,7.5800,61.9900,4.6300
4,DPT005,Sales,Technology,702,"8,214,616.0600",26,29.5900,8.0900,58.8600,3.7000
10,DPT011,Logistics,Technology,483,"6,110,813.4900",24,34.2200,7.7100,71.1900,4.9700
7,DPT008,Legal,Operations,399,"5,717,785.4000",23,33.5400,7.7200,64.8100,5.7600



Anomaly summary files saved successfully:
- ..\reports\supplier_anomaly_summary.csv
- ..\reports\department_anomaly_summary.csv


In [17]:
# ============================================================
# 11. ANOMALY DETECTION VISUALIZATIONS
# ============================================================

# ------------------------------------------------------------
# Anomaly distribution chart
# ------------------------------------------------------------

fig_anomaly_distribution = px.bar(
    anomaly_distribution,
    x="Label",
    y="Invoices",
    text="Invoices",
    title="Machine Learning Anomaly Detection Distribution",
    labels={
        "Label": "Transaction Type",
        "Invoices": "Number of Invoices"
    }
)

fig_anomaly_distribution.update_traces(textposition="outside")
fig_anomaly_distribution.update_layout(height=420)
fig_anomaly_distribution.show()

# ------------------------------------------------------------
# Audit risk score vs ML anomaly score
# ------------------------------------------------------------

fig_audit_ml_scatter = px.scatter(
    ml_df,
    x="audit_risk_score",
    y="ml_anomaly_score",
    color="ml_anomaly_flag",
    size="invoice_amount",
    hover_name="invoice_id",
    hover_data=[
        "supplier_name",
        "department_name",
        "invoice_amount",
        "audit_risk_level",
        "ml_review_priority",
        "primary_risk_driver"
    ],
    title="Audit Risk Score vs Machine Learning Anomaly Score",
    labels={
        "audit_risk_score": "Audit Rules Risk Score",
        "ml_anomaly_score": "ML Anomaly Score",
        "ml_anomaly_flag": "ML Anomaly Flag"
    }
)

fig_audit_ml_scatter.update_layout(height=560)
fig_audit_ml_scatter.show()

# ------------------------------------------------------------
# Top departments by anomaly count
# ------------------------------------------------------------

fig_department_anomalies = px.bar(
    department_anomaly_summary.sort_values("Anomaly_Count", ascending=True),
    x="Anomaly_Count",
    y="department_name",
    color="business_unit",
    orientation="h",
    title="Detected Anomalies by Department",
    labels={
        "Anomaly_Count": "Detected Anomalies",
        "department_name": "Department",
        "business_unit": "Business Unit"
    }
)

fig_department_anomalies.update_layout(height=520)
fig_department_anomalies.show()

# ------------------------------------------------------------
# Top suppliers by anomaly count
# ------------------------------------------------------------

fig_supplier_anomalies = px.bar(
    supplier_anomaly_summary.head(15).sort_values("Anomaly_Count", ascending=True),
    x="Anomaly_Count",
    y="supplier_name",
    color="supplier_category",
    orientation="h",
    title="Top 15 Suppliers by Detected Anomalies",
    labels={
        "Anomaly_Count": "Detected Anomalies",
        "supplier_name": "Supplier",
        "supplier_category": "Supplier Category"
    }
)

fig_supplier_anomalies.update_layout(height=560)
fig_supplier_anomalies.show()

# 9. SQL Audit Analytics Data Mart

This section creates a SQLite audit analytics data mart.

The objective is to transform the notebook outputs into a structured database that can support SQL-based audit analysis, dashboarding and reporting.

The data mart includes:

- fact audit transactions;
- supplier dimension;
- department dimension;
- purchase order dimension;
- audit rules summary;
- supplier risk summary;
- machine learning anomaly detection results.

This demonstrates the ability to move from raw data analysis to a structured analytics layer, which is important for audit, risk advisory and financial data analytics roles.

In [18]:
# ============================================================
# 9. SQL AUDIT ANALYTICS DATA MART
# ============================================================

print("=" * 80)
print("SQL AUDIT ANALYTICS DATA MART")
print("=" * 80)

import sqlite3

# ------------------------------------------------------------
# Database path
# ------------------------------------------------------------

db_path = PROCESSED_DATA_DIR / "financial_audit_risk_analytics.db"

# Remove old database if it exists
if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(db_path)

# ------------------------------------------------------------
# Prepare tables for SQL export
# ------------------------------------------------------------

fact_audit_transactions = ml_df.copy()

dim_suppliers = suppliers_clean.copy()
dim_departments = departments_clean.copy()
dim_purchase_orders = purchase_orders_clean.copy()
dim_payments = payments_clean.copy()

# Keep useful reporting tables
sql_audit_rules_summary = audit_rules_summary_df.copy()
sql_audit_category_summary = audit_category_summary.copy()
sql_supplier_risk_summary = supplier_risk_summary.copy()
sql_executive_audit_kpis = executive_audit_kpis.copy()
sql_department_risk_summary = department_risk_summary.copy()
sql_business_unit_risk_summary = business_unit_risk_summary.copy()
sql_monthly_audit_trend = monthly_audit_trend.copy()
sql_supplier_category_risk_summary = supplier_category_risk_summary.copy()
sql_ml_anomaly_distribution = anomaly_distribution.copy()
sql_top_anomalous_transactions = top_anomalous_transactions.copy()
sql_supplier_anomaly_summary = supplier_anomaly_summary.copy()
sql_department_anomaly_summary = department_anomaly_summary.copy()

# ------------------------------------------------------------
# Write tables to SQLite
# ------------------------------------------------------------

fact_audit_transactions.to_sql(
    "fact_audit_transactions",
    conn,
    index=False,
    if_exists="replace"
)

dim_suppliers.to_sql(
    "dim_suppliers",
    conn,
    index=False,
    if_exists="replace"
)

dim_departments.to_sql(
    "dim_departments",
    conn,
    index=False,
    if_exists="replace"
)

dim_purchase_orders.to_sql(
    "dim_purchase_orders",
    conn,
    index=False,
    if_exists="replace"
)

dim_payments.to_sql(
    "dim_payments",
    conn,
    index=False,
    if_exists="replace"
)

sql_audit_rules_summary.to_sql(
    "audit_rules_summary",
    conn,
    index=False,
    if_exists="replace"
)

sql_audit_category_summary.to_sql(
    "audit_category_summary",
    conn,
    index=False,
    if_exists="replace"
)

sql_supplier_risk_summary.to_sql(
    "supplier_risk_summary",
    conn,
    index=False,
    if_exists="replace"
)

sql_executive_audit_kpis.to_sql(
    "executive_audit_kpis",
    conn,
    index=False,
    if_exists="replace"
)

sql_department_risk_summary.to_sql(
    "department_risk_summary",
    conn,
    index=False,
    if_exists="replace"
)

sql_business_unit_risk_summary.to_sql(
    "business_unit_risk_summary",
    conn,
    index=False,
    if_exists="replace"
)

sql_monthly_audit_trend.to_sql(
    "monthly_audit_risk_trend",
    conn,
    index=False,
    if_exists="replace"
)

sql_supplier_category_risk_summary.to_sql(
    "supplier_category_risk_summary",
    conn,
    index=False,
    if_exists="replace"
)

sql_ml_anomaly_distribution.to_sql(
    "ml_anomaly_distribution",
    conn,
    index=False,
    if_exists="replace"
)

sql_top_anomalous_transactions.to_sql(
    "top_anomalous_transactions",
    conn,
    index=False,
    if_exists="replace"
)

sql_supplier_anomaly_summary.to_sql(
    "supplier_anomaly_summary",
    conn,
    index=False,
    if_exists="replace"
)

sql_department_anomaly_summary.to_sql(
    "department_anomaly_summary",
    conn,
    index=False,
    if_exists="replace"
)

# ------------------------------------------------------------
# Create SQL views
# ------------------------------------------------------------

cursor = conn.cursor()

cursor.execute("""
DROP VIEW IF EXISTS vw_high_priority_transactions;
""")

cursor.execute("""
CREATE VIEW vw_high_priority_transactions AS
SELECT
    invoice_id,
    invoice_number,
    supplier_id,
    supplier_name,
    supplier_category,
    supplier_country,
    department_id,
    department_name,
    business_unit,
    invoice_date,
    invoice_amount,
    currency,
    audit_rule_count,
    audit_risk_score,
    audit_risk_level,
    primary_risk_driver,
    ml_anomaly_flag,
    ml_anomaly_score,
    combined_review_priority_score,
    ml_review_priority
FROM fact_audit_transactions
WHERE audit_risk_level IN ('High Risk', 'Critical Risk')
   OR ml_review_priority IN ('High Review Priority', 'Critical Review Priority')
ORDER BY combined_review_priority_score DESC;
""")

cursor.execute("""
DROP VIEW IF EXISTS vw_supplier_audit_priorities;
""")

cursor.execute("""
CREATE VIEW vw_supplier_audit_priorities AS
SELECT
    supplier_id,
    supplier_name,
    supplier_category,
    supplier_country,
    Invoice_Count,
    Total_Invoice_Amount,
    Spend_Share,
    Exception_Rate,
    High_Risk_Invoice_Rate,
    Critical_Rule_Triggers,
    supplier_risk_score,
    supplier_risk_level,
    primary_supplier_risk_driver,
    Supplier_Audit_Recommendation
FROM supplier_risk_summary
ORDER BY supplier_risk_score DESC;
""")

cursor.execute("""
DROP VIEW IF EXISTS vw_department_audit_risk;
""")

cursor.execute("""
CREATE VIEW vw_department_audit_risk AS
SELECT
    department_id,
    department_name,
    business_unit,
    Invoice_Count,
    Total_Invoice_Amount,
    Exception_Invoices,
    Average_Risk_Score,
    High_Risk_Invoices,
    Exception_Rate,
    High_Risk_Rate,
    Spend_Share
FROM department_risk_summary
ORDER BY Average_Risk_Score DESC;
""")

conn.commit()

# ------------------------------------------------------------
# Validate database tables
# ------------------------------------------------------------

tables_created = pd.read_sql_query(
    """
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
    ORDER BY type, name;
    """,
    conn
)

print("SQLite data mart created successfully")
print("Database path:", db_path)

print("\nTables and views created:")
display(tables_created)

# ------------------------------------------------------------
# Quick SQL validation queries
# ------------------------------------------------------------

sql_global_kpis = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_invoices,
        ROUND(SUM(invoice_amount), 2) AS total_invoice_amount,
        ROUND(AVG(audit_risk_score), 2) AS average_audit_risk_score,
        SUM(CASE WHEN audit_rule_count > 0 THEN 1 ELSE 0 END) AS invoices_with_exceptions,
        ROUND(
            SUM(CASE WHEN audit_rule_count > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS exception_rate_percentage,
        SUM(ml_anomaly_flag) AS ml_detected_anomalies,
        ROUND(SUM(ml_anomaly_flag) * 100.0 / COUNT(*), 2) AS ml_anomaly_rate_percentage
    FROM fact_audit_transactions;
    """,
    conn
)

sql_top_priority_transactions = pd.read_sql_query(
    """
    SELECT *
    FROM vw_high_priority_transactions
    LIMIT 10;
    """,
    conn
)

sql_top_supplier_priorities = pd.read_sql_query(
    """
    SELECT *
    FROM vw_supplier_audit_priorities
    LIMIT 10;
    """,
    conn
)

print("\nSQL global KPIs:")
display(sql_global_kpis)

print("\nSQL top priority transactions:")
display(sql_top_priority_transactions)

print("\nSQL top supplier audit priorities:")
display(sql_top_supplier_priorities)

conn.close()

# ------------------------------------------------------------
# Save SQL scripts
# ------------------------------------------------------------

schema_sql = """
-- ============================================================
-- Financial Audit Risk Analytics Platform
-- SQLite Data Mart Schema
-- ============================================================

-- Core fact table
SELECT
    invoice_id,
    invoice_number,
    supplier_id,
    department_id,
    invoice_date,
    invoice_amount,
    currency,
    audit_rule_count,
    audit_risk_score,
    audit_risk_level,
    ml_anomaly_flag,
    ml_anomaly_score,
    combined_review_priority_score,
    ml_review_priority
FROM fact_audit_transactions
LIMIT 10;

-- Supplier dimension
SELECT *
FROM dim_suppliers
LIMIT 10;

-- Department dimension
SELECT *
FROM dim_departments
LIMIT 10;

-- Purchase order dimension
SELECT *
FROM dim_purchase_orders
LIMIT 10;
"""

executive_queries_sql = """
-- ============================================================
-- Executive Audit KPI Queries
-- ============================================================

-- Global audit KPIs
SELECT
    COUNT(*) AS total_invoices,
    ROUND(SUM(invoice_amount), 2) AS total_invoice_amount,
    ROUND(AVG(audit_risk_score), 2) AS average_audit_risk_score,
    SUM(CASE WHEN audit_rule_count > 0 THEN 1 ELSE 0 END) AS invoices_with_exceptions,
    ROUND(
        SUM(CASE WHEN audit_rule_count > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS exception_rate_percentage,
    SUM(CASE WHEN audit_risk_level IN ('High Risk', 'Critical Risk') THEN 1 ELSE 0 END) AS high_risk_invoices,
    SUM(ml_anomaly_flag) AS ml_detected_anomalies
FROM fact_audit_transactions;

-- Risk level distribution
SELECT
    audit_risk_level,
    COUNT(*) AS invoices,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_audit_transactions), 2) AS percentage
FROM fact_audit_transactions
GROUP BY audit_risk_level
ORDER BY invoices DESC;

-- Monthly audit risk trend
SELECT
    invoice_year,
    invoice_month,
    COUNT(*) AS invoice_count,
    ROUND(SUM(invoice_amount), 2) AS total_invoice_amount,
    ROUND(AVG(audit_risk_score), 2) AS average_risk_score,
    SUM(CASE WHEN audit_rule_count > 0 THEN 1 ELSE 0 END) AS exception_invoices
FROM fact_audit_transactions
GROUP BY invoice_year, invoice_month
ORDER BY invoice_year, invoice_month;
"""

supplier_queries_sql = """
-- ============================================================
-- Supplier Risk Queries
-- ============================================================

-- Top supplier audit priorities
SELECT *
FROM vw_supplier_audit_priorities
LIMIT 20;

-- High-risk suppliers
SELECT
    supplier_id,
    supplier_name,
    supplier_category,
    supplier_country,
    Invoice_Count,
    Total_Invoice_Amount,
    supplier_risk_score,
    supplier_risk_level,
    primary_supplier_risk_driver
FROM supplier_risk_summary
WHERE supplier_risk_level = 'High Supplier Risk'
ORDER BY supplier_risk_score DESC;

-- Suppliers by ML anomaly count
SELECT
    supplier_id,
    supplier_name,
    supplier_category,
    supplier_country,
    Invoice_Count,
    Anomaly_Count,
    Anomaly_Rate,
    Average_Anomaly_Score
FROM supplier_anomaly_summary
ORDER BY Anomaly_Count DESC
LIMIT 20;
"""

exception_queries_sql = """
-- ============================================================
-- Audit Exception Queries
-- ============================================================

-- High priority transactions
SELECT *
FROM vw_high_priority_transactions
LIMIT 25;

-- Department risk ranking
SELECT *
FROM vw_department_audit_risk;

-- Audit rules summary
SELECT *
FROM audit_rules_summary
ORDER BY Risk_Weight DESC, Flagged_Invoices DESC;

-- Machine learning anomaly distribution
SELECT *
FROM ml_anomaly_distribution;
"""

schema_path = SQL_DIR / "01_data_mart_schema.sql"
executive_queries_path = SQL_DIR / "02_executive_kpi_queries.sql"
supplier_queries_path = SQL_DIR / "03_supplier_risk_queries.sql"
exception_queries_path = SQL_DIR / "04_audit_exception_queries.sql"

schema_path.write_text(schema_sql, encoding="utf-8")
executive_queries_path.write_text(executive_queries_sql, encoding="utf-8")
supplier_queries_path.write_text(supplier_queries_sql, encoding="utf-8")
exception_queries_path.write_text(exception_queries_sql, encoding="utf-8")

print("\nSQL files created successfully:")
print("-", schema_path)
print("-", executive_queries_path)
print("-", supplier_queries_path)
print("-", exception_queries_path)

SQL AUDIT ANALYTICS DATA MART
SQLite data mart created successfully
Database path: ..\data\processed\financial_audit_risk_analytics.db

Tables and views created:


,name,type
0,audit_category_summary,table
1,audit_rules_summary,table
2,business_unit_risk_summary,table
3,department_anomaly_summary,table
4,department_risk_summary,table
5,dim_departments,table
6,dim_payments,table
7,dim_purchase_orders,table
8,dim_suppliers,table
9,executive_audit_kpis,table



SQL global KPIs:


,total_invoices,total_invoice_amount,average_audit_risk_score,invoices_with_exceptions,exception_rate_percentage,ml_detected_anomalies,ml_anomaly_rate_percentage
0,8630,"111,360,795.8300",7.9000,7116,82.4600,432,5.0100



SQL top priority transactions:


,invoice_id,invoice_number,supplier_id,supplier_name,supplier_category,supplier_country,department_id,department_name,business_unit,invoice_date,invoice_amount,currency,audit_rule_count,audit_risk_score,audit_risk_level,primary_risk_driver,ml_anomaly_flag,ml_anomaly_score,combined_review_priority_score,ml_review_priority
0,INV0006305,FCT-360582,SUP00101,Supplier_0101,Professional Services,United Arab Emirates,DPT011,Logistics,Technology,2024-07-08 00:00:00,"100,000.0000",DZD,6,30.1100,High Risk,R13 - PO department mismatch,1,98.5800,71.1900,Critical Review Priority
1,INV0002035,FCT-528008,SUP00002,Supplier_0002,Training,Germany,DPT009,Risk Management,Commercial,2024-09-18 00:00:00,"100,000.0000",DZD,6,30.1100,High Risk,R15 - PO not approved,1,94.4500,68.7100,High Review Priority
2,INV0001101,FCT-253802,SUP00018,Supplier_0018,Marketing Services,Algeria,DPT004,Operations,Technology,2024-04-05 00:00:00,"106,166.2600",EUR,4,21.5100,Medium Risk,R15 - PO not approved,1,99.8800,68.5300,High Review Priority
3,INV0006516,FCT-701982,SUP00074,Supplier_0074,Telecom,United Arab Emirates,DPT006,Marketing,Support,2024-06-19 00:00:00,"138,721.0300",DZD,6,33.8700,High Risk,R06 - Payment before invoice date,1,88.0000,66.3500,High Review Priority
4,INV0006588,FCT-546673,SUP00016,Supplier_0016,Consulting,United Arab Emirates,DPT003,IT,Operations,2024-12-27 00:00:00,"207,315.9900",EUR,4,17.7400,Medium Risk,R04 - High-value invoice,1,96.8900,65.2300,High Review Priority
5,INV0008590,FCT-533287,SUP00123,Supplier_0123,Maintenance,Germany,DPT008,Legal,Operations,2024-11-24 00:00:00,706.9100,EUR,2,16.1300,Medium Risk,R02 - Potential duplicate invoice,1,97.2600,64.8100,High Review Priority
6,INV0004163,FCT-759856,SUP00003,Supplier_0003,Office Supplies,Turkey,DPT006,Marketing,Support,2024-07-17 00:00:00,"96,009.3000",DZD,4,18.2800,Medium Risk,R13 - PO department mismatch,1,95.7000,64.7300,High Review Priority
7,INV0007814,FCT-396205,SUP00006,Supplier_0006,Telecom,Algeria,DPT002,Procurement,Operations,2024-12-09 00:00:00,"100,000.0000",DZD,5,27.9600,Medium Risk,R02 - Potential duplicate invoice,1,88.4300,64.2400,High Review Priority
8,INV0002336,FCT-891175,SUP00003,Supplier_0003,Office Supplies,Turkey,DPT012,General Administration,Support,2024-09-17 00:00:00,"41,780.4400",EUR,2,9.6800,Low Risk,R01 - Invoice without purchase order,1,100.0000,63.8700,High Review Priority
9,INV0001590,FCT-458849,SUP00281,Supplier_0281,Telecom,Germany,DPT006,Marketing,Support,2024-12-17 00:00:00,"66,438.9100",EUR,3,22.5800,Medium Risk,R12 - PO supplier mismatch,1,90.8800,63.5600,High Review Priority



SQL top supplier audit priorities:


,supplier_id,supplier_name,supplier_category,supplier_country,Invoice_Count,Total_Invoice_Amount,Spend_Share,Exception_Rate,High_Risk_Invoice_Rate,Critical_Rule_Triggers,supplier_risk_score,supplier_risk_level,primary_supplier_risk_driver,Supplier_Audit_Recommendation
0,SUP00184,Supplier_0184,Facilities,Algeria,8,"197,372.2300",0.1800,100.0000,12.5000,2,66.5900,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0184. Mai...
1,SUP00301,Supplier_0301,Facilities,France,17,"123,829.9300",0.1100,100.0000,5.8800,4,58.4200,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0301. Mai...
2,SUP00021,Supplier_0021,Maintenance,Algeria,119,"1,462,405.7100",1.3100,100.0000,1.6800,16,58.2100,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0021. Mai...
3,SUP00017,Supplier_0017,Logistics,Algeria,79,"1,049,765.6800",0.9400,100.0000,1.2700,9,55.6400,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0017. Mai...
4,SUP00147,Supplier_0147,Marketing Services,United Arab Emirates,17,"177,322.6200",0.1600,100.0000,5.8800,2,54.3300,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0147. Mai...
5,SUP00216,Supplier_0216,Marketing Services,Spain,7,"77,439.1900",0.0700,100.0000,0.0000,2,53.8600,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0216. Mai...
6,SUP00281,Supplier_0281,Telecom,Germany,8,"300,452.5100",0.2700,100.0000,0.0000,2,53.7600,High Supplier Risk,High average transaction risk,Prioritize audit review for Supplier_0281. Mai...
7,SUP00106,Supplier_0106,Office Supplies,Algeria,3,"37,655.5300",0.0300,100.0000,0.0000,1,53.1700,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0106. Mai...
8,SUP00009,Supplier_0009,IT Services,Algeria,325,"4,126,869.9800",3.7100,85.2300,0.0000,34,53.0200,High Supplier Risk,Critical audit rule triggers,Prioritize audit review for Supplier_0009. Mai...
9,SUP00030,Supplier_0030,Logistics,France,102,"1,276,173.3400",1.1500,100.0000,0.0000,9,51.9300,High Supplier Risk,High exception rate,Prioritize audit review for Supplier_0030. Mai...



SQL files created successfully:
- ..\sql\01_data_mart_schema.sql
- ..\sql\02_executive_kpi_queries.sql
- ..\sql\03_supplier_risk_queries.sql
- ..\sql\04_audit_exception_queries.sql


# 10. Automated Excel Audit Report

This section generates a professional Excel audit report.

The Excel workbook summarizes the main audit analytics outputs into a structured client-style deliverable.

The report includes:

- executive audit KPIs;
- audit risk level distribution;
- audit rules summary;
- audit category summary;
- high-priority transactions;
- supplier risk ranking;
- department risk summary;
- monthly audit risk trend;
- machine learning anomaly detection results.

This deliverable demonstrates automated reporting skills for audit, financial risk analytics and consulting projects.

In [19]:
# ============================================================
# 10. AUTOMATED EXCEL AUDIT REPORT
# ============================================================

print("=" * 80)
print("AUTOMATED EXCEL AUDIT REPORT")
print("=" * 80)

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.chart import BarChart, LineChart, PieChart, Reference
from openpyxl.worksheet.table import Table, TableStyleInfo

# ------------------------------------------------------------
# Excel output path
# ------------------------------------------------------------

excel_report_path = REPORTS_DIR / "financial_audit_risk_analytics_report.xlsx"

# ------------------------------------------------------------
# Prepare selected outputs
# ------------------------------------------------------------

top_supplier_risk_excel = supplier_risk_summary[
    [
        "supplier_id",
        "supplier_name",
        "supplier_category",
        "supplier_country",
        "Invoice_Count",
        "Total_Invoice_Amount",
        "Spend_Share",
        "Exception_Rate",
        "High_Risk_Invoice_Rate",
        "Critical_Rule_Triggers",
        "supplier_risk_score",
        "supplier_risk_level",
        "primary_supplier_risk_driver"
    ]
].head(50)

top_priority_transactions_excel = ml_df.sort_values(
    ["combined_review_priority_score", "ml_anomaly_score", "audit_risk_score"],
    ascending=False
)[
    [
        "invoice_id",
        "invoice_number",
        "supplier_name",
        "department_name",
        "supplier_category",
        "invoice_date",
        "invoice_amount",
        "currency",
        "audit_rule_count",
        "audit_risk_score",
        "audit_risk_level",
        "ml_anomaly_flag",
        "ml_anomaly_score",
        "combined_review_priority_score",
        "ml_review_priority",
        "primary_risk_driver"
    ]
].head(100)

top_anomalies_excel = top_anomalous_transactions.copy()

# ------------------------------------------------------------
# Write sheets
# ------------------------------------------------------------

with pd.ExcelWriter(excel_report_path, engine="openpyxl") as writer:
    executive_audit_kpis.to_excel(writer, sheet_name="Executive KPIs", index=False)
    risk_level_distribution.to_excel(writer, sheet_name="Risk Distribution", index=False)
    audit_rules_summary_df.to_excel(writer, sheet_name="Audit Rules", index=False)
    audit_category_summary.to_excel(writer, sheet_name="Audit Categories", index=False)
    top_priority_transactions_excel.to_excel(writer, sheet_name="Priority Transactions", index=False)
    top_supplier_risk_excel.to_excel(writer, sheet_name="Supplier Risk", index=False)
    department_risk_summary.to_excel(writer, sheet_name="Department Risk", index=False)
    business_unit_risk_summary.to_excel(writer, sheet_name="Business Unit Risk", index=False)
    monthly_audit_trend.to_excel(writer, sheet_name="Monthly Trend", index=False)
    supplier_category_risk_summary.to_excel(writer, sheet_name="Supplier Categories", index=False)
    anomaly_distribution.to_excel(writer, sheet_name="ML Anomaly Summary", index=False)
    top_anomalies_excel.to_excel(writer, sheet_name="Top ML Anomalies", index=False)

print("Excel report created successfully")
print("Excel report path:", excel_report_path)

AUTOMATED EXCEL AUDIT REPORT
Excel report created successfully
Excel report path: ..\reports\financial_audit_risk_analytics_report.xlsx


In [20]:
# ============================================================
# 10.1 EXCEL REPORT FORMATTING
# ============================================================

wb = load_workbook(excel_report_path)

# ------------------------------------------------------------
# Style configuration
# ------------------------------------------------------------

dark_blue_fill = PatternFill("solid", fgColor="1F4E78")
medium_blue_fill = PatternFill("solid", fgColor="5B9BD5")
light_blue_fill = PatternFill("solid", fgColor="D9EAF7")
light_green_fill = PatternFill("solid", fgColor="E2F0D9")
light_orange_fill = PatternFill("solid", fgColor="FCE4D6")
light_red_fill = PatternFill("solid", fgColor="F4CCCC")

white_font = Font(color="FFFFFF", bold=True)
title_font = Font(color="1F4E78", bold=True, size=14)
header_font = Font(color="FFFFFF", bold=True)
body_font = Font(size=10)

thin_border = Border(
    left=Side(style="thin", color="D9E2F3"),
    right=Side(style="thin", color="D9E2F3"),
    top=Side(style="thin", color="D9E2F3"),
    bottom=Side(style="thin", color="D9E2F3")
)

# ------------------------------------------------------------
# General sheet formatting
# ------------------------------------------------------------

for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    
    # Header formatting
    for cell in ws[1]:
        cell.fill = dark_blue_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = thin_border
    
    # Body formatting
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font = body_font
            cell.alignment = Alignment(vertical="center", wrap_text=True)
            cell.border = thin_border
    
    # Auto width
    for column_cells in ws.columns:
        max_length = 0
        column_letter = get_column_letter(column_cells[0].column)
        
        for cell in column_cells:
            try:
                value_length = len(str(cell.value)) if cell.value is not None else 0
                max_length = max(max_length, value_length)
            except:
                pass
        
        adjusted_width = min(max(max_length + 2, 12), 35)
        ws.column_dimensions[column_letter].width = adjusted_width
    
    # Row height
    ws.row_dimensions[1].height = 28
    
    # Add filters
    ws.auto_filter.ref = ws.dimensions

# ------------------------------------------------------------
# Number formatting
# ------------------------------------------------------------

amount_keywords = [
    "Amount",
    "Exposure",
    "Revenue",
    "Spend",
    "Total_Invoice_Amount",
    "Average_Invoice_Amount"
]

percentage_keywords = [
    "Rate",
    "Share",
    "Percentage"
]

score_keywords = [
    "Score",
    "score"
]

for ws in wb.worksheets:
    headers = [cell.value for cell in ws[1]]
    
    for col_idx, header in enumerate(headers, start=1):
        if header is None:
            continue
        
        col_letter = get_column_letter(col_idx)
        
        if any(keyword in str(header) for keyword in amount_keywords):
            for cell in ws[col_letter][1:]:
                cell.number_format = '#,##0.00'
        
        if any(keyword in str(header) for keyword in percentage_keywords):
            for cell in ws[col_letter][1:]:
                cell.number_format = '0.00'
        
        if any(keyword in str(header) for keyword in score_keywords):
            for cell in ws[col_letter][1:]:
                cell.number_format = '0.00'

# ------------------------------------------------------------
# Conditional formatting-like fills for risk levels
# ------------------------------------------------------------

risk_sheets = ["Priority Transactions", "Supplier Risk", "Department Risk", "Top ML Anomalies"]

for sheet_name in risk_sheets:
    if sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        headers = [cell.value for cell in ws[1]]
        
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value, str):
                    value = cell.value
                    
                    if "High" in value or "Critical" in value:
                        cell.fill = light_red_fill
                    elif "Medium" in value:
                        cell.fill = light_orange_fill
                    elif "Low" in value:
                        cell.fill = light_green_fill

# ------------------------------------------------------------
# Add Excel tables
# ------------------------------------------------------------

for ws in wb.worksheets:
    if ws.max_row > 1 and ws.max_column > 1:
        table_ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
        table_name = ws.title.replace(" ", "_").replace("-", "_")[:25]
        
        # Avoid duplicate table names
        table_name = f"T_{table_name}"
        
        tab = Table(displayName=table_name, ref=table_ref)
        style = TableStyleInfo(
            name="TableStyleMedium2",
            showFirstColumn=False,
            showLastColumn=False,
            showRowStripes=True,
            showColumnStripes=False
        )
        tab.tableStyleInfo = style
        
        try:
            ws.add_table(tab)
        except:
            pass

# ------------------------------------------------------------
# Add charts to key sheets
# ------------------------------------------------------------

# Risk distribution chart
if "Risk Distribution" in wb.sheetnames:
    ws = wb["Risk Distribution"]
    
    chart = BarChart()
    chart.title = "Audit Risk Level Distribution"
    chart.y_axis.title = "Invoices"
    chart.x_axis.title = "Risk Level"
    
    data = Reference(ws, min_col=2, min_row=1, max_row=ws.max_row)
    cats = Reference(ws, min_col=1, min_row=2, max_row=ws.max_row)
    
    chart.add_data(data, titles_from_data=True)
    chart.set_categories(cats)
    chart.height = 8
    chart.width = 14
    
    ws.add_chart(chart, "E2")

# Monthly trend chart
if "Monthly Trend" in wb.sheetnames:
    ws = wb["Monthly Trend"]
    
    headers = [cell.value for cell in ws[1]]
    
    if "Average_Risk_Score" in headers:
        avg_score_col = headers.index("Average_Risk_Score") + 1
        month_col = headers.index("Month") + 1
        
        chart = LineChart()
        chart.title = "Monthly Average Audit Risk Score"
        chart.y_axis.title = "Average Risk Score"
        chart.x_axis.title = "Month"
        
        data = Reference(ws, min_col=avg_score_col, min_row=1, max_row=ws.max_row)
        cats = Reference(ws, min_col=month_col, min_row=2, max_row=ws.max_row)
        
        chart.add_data(data, titles_from_data=True)
        chart.set_categories(cats)
        chart.height = 8
        chart.width = 16
        
        ws.add_chart(chart, "N2")

# Supplier risk chart
if "Supplier Risk" in wb.sheetnames:
    ws = wb["Supplier Risk"]
    
    headers = [cell.value for cell in ws[1]]
    
    if "supplier_risk_score" in headers and "supplier_name" in headers:
        score_col = headers.index("supplier_risk_score") + 1
        supplier_col = headers.index("supplier_name") + 1
        
        chart = BarChart()
        chart.title = "Top Supplier Risk Scores"
        chart.y_axis.title = "Supplier Risk Score"
        chart.x_axis.title = "Supplier"
        
        data = Reference(ws, min_col=score_col, min_row=1, max_row=min(ws.max_row, 11))
        cats = Reference(ws, min_col=supplier_col, min_row=2, max_row=min(ws.max_row, 11))
        
        chart.add_data(data, titles_from_data=True)
        chart.set_categories(cats)
        chart.height = 8
        chart.width = 16
        
        ws.add_chart(chart, "Q2")

# ML anomaly chart
if "ML Anomaly Summary" in wb.sheetnames:
    ws = wb["ML Anomaly Summary"]
    
    chart = PieChart()
    chart.title = "ML Anomaly Detection Distribution"
    
    data = Reference(ws, min_col=2, min_row=1, max_row=ws.max_row)
    cats = Reference(ws, min_col=3, min_row=2, max_row=ws.max_row)
    
    chart.add_data(data, titles_from_data=True)
    chart.set_categories(cats)
    chart.height = 8
    chart.width = 12
    
    ws.add_chart(chart, "F2")

# ------------------------------------------------------------
# Reorder sheets
# ------------------------------------------------------------

preferred_order = [
    "Executive KPIs",
    "Risk Distribution",
    "Audit Rules",
    "Audit Categories",
    "Priority Transactions",
    "Supplier Risk",
    "Department Risk",
    "Business Unit Risk",
    "Monthly Trend",
    "Supplier Categories",
    "ML Anomaly Summary",
    "Top ML Anomalies"
]

ordered_sheets = [wb[s] for s in preferred_order if s in wb.sheetnames]
remaining_sheets = [ws for ws in wb.worksheets if ws.title not in preferred_order]

wb._sheets = ordered_sheets + remaining_sheets

# ------------------------------------------------------------
# Save final formatted workbook
# ------------------------------------------------------------

wb.save(excel_report_path)

print("Excel report formatted successfully")
print("Final Excel report path:", excel_report_path)

Excel report formatted successfully
Final Excel report path: ..\reports\financial_audit_risk_analytics_report.xlsx


# 11. Interactive HTML Audit Dashboard

This section creates a professional interactive HTML dashboard for the Financial Audit Risk Analytics Platform.

The dashboard summarizes the main outputs of the project:

- executive audit KPIs;
- audit risk level distribution;
- audit rules summary;
- supplier risk ranking;
- department risk analysis;
- monthly audit risk trend;
- machine learning anomaly detection;
- high-priority transactions;
- business recommendations.

The dashboard is designed as a portfolio-ready deliverable that can be published with GitHub Pages.

In [21]:
# ============================================================
# 11. INTERACTIVE HTML AUDIT DASHBOARD
# ============================================================

print("=" * 80)
print("INTERACTIVE HTML AUDIT DASHBOARD")
print("=" * 80)

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

# ------------------------------------------------------------
# Dashboard output path
# ------------------------------------------------------------

dashboard_path = DASHBOARD_DIR / "index.html"
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def get_kpi_value(kpi_name):
    value = executive_audit_kpis.loc[
        executive_audit_kpis["KPI"] == kpi_name,
        "Value"
    ].values
    
    if len(value) > 0:
        return value[0]
    return "N/A"


def format_number(value):
    try:
        return f"{float(value):,.0f}"
    except:
        return value


def format_amount(value):
    try:
        return f"{float(value):,.2f}"
    except:
        return value


def plot_to_html(fig):
    return pio.to_html(
        fig,
        full_html=False,
        include_plotlyjs=False,
        config={"displayModeBar": True, "responsive": True}
    )

# ------------------------------------------------------------
# KPI values
# ------------------------------------------------------------

total_invoices_kpi = get_kpi_value("Total Invoices")
total_amount_kpi = get_kpi_value("Total Invoice Amount")
exception_rate_kpi = get_kpi_value("Audit Exception Rate (%)")
high_risk_invoices_kpi = get_kpi_value("High/Critical Risk Invoices")
high_risk_exposure_kpi = get_kpi_value("High/Critical Risk Exposure")
ml_anomalies_kpi = int(ml_df["ml_anomaly_flag"].sum())
high_risk_suppliers_kpi = get_kpi_value("High-Risk Suppliers")
top_supplier_kpi = get_kpi_value("Top Risk Supplier")

# ------------------------------------------------------------
# Figure 1: Risk level distribution
# ------------------------------------------------------------

fig_risk_distribution = px.bar(
    risk_level_distribution,
    x="Audit_Risk_Level",
    y="Invoices",
    text="Invoices",
    title="Audit Risk Level Distribution",
    color="Audit_Risk_Level"
)

fig_risk_distribution.update_traces(textposition="outside")
fig_risk_distribution.update_layout(
    height=430,
    showlegend=False,
    margin=dict(l=40, r=40, t=70, b=40)
)

# ------------------------------------------------------------
# Figure 2: Audit rules summary
# ------------------------------------------------------------

fig_audit_rules = px.bar(
    audit_rules_summary_df.sort_values("Flagged_Invoices", ascending=True),
    x="Flagged_Invoices",
    y="Rule_Name",
    orientation="h",
    color="Severity",
    title="Audit Rules Triggered by Number of Invoices",
    labels={
        "Flagged_Invoices": "Flagged Invoices",
        "Rule_Name": "Audit Rule"
    }
)

fig_audit_rules.update_layout(
    height=650,
    margin=dict(l=40, r=40, t=70, b=40)
)

# ------------------------------------------------------------
# Figure 3: Audit category summary
# ------------------------------------------------------------

fig_audit_categories = px.bar(
    audit_category_summary.sort_values("Exceptions", ascending=True),
    x="Exceptions",
    y="Risk_Category",
    orientation="h",
    color="Average_Risk_Score",
    title="Audit Exceptions by Risk Category",
    labels={
        "Exceptions": "Number of Exceptions",
        "Risk_Category": "Risk Category",
        "Average_Risk_Score": "Avg. Risk Score"
    }
)

fig_audit_categories.update_layout(
    height=500,
    margin=dict(l=40, r=40, t=70, b=40)
)

# ------------------------------------------------------------
# Figure 4: Supplier risk ranking
# ------------------------------------------------------------

fig_supplier_risk = px.bar(
    supplier_risk_summary.head(15).sort_values("supplier_risk_score", ascending=True),
    x="supplier_risk_score",
    y="supplier_name",
    orientation="h",
    color="supplier_risk_level",
    title="Top 15 Suppliers by Risk Score",
    labels={
        "supplier_risk_score": "Supplier Risk Score",
        "supplier_name": "Supplier",
        "supplier_risk_level": "Risk Level"
    }
)

fig_supplier_risk.update_layout(
    height=560,
    margin=dict(l=40, r=40, t=70, b=40)
)

# ------------------------------------------------------------
# Figure 5: Department risk summary
# ------------------------------------------------------------

fig_department_risk = px.bar(
    department_risk_summary.sort_values("Average_Risk_Score", ascending=True),
    x="Average_Risk_Score",
    y="department_name",
    orientation="h",
    color="business_unit",
    title="Average Audit Risk Score by Department",
    labels={
        "Average_Risk_Score": "Average Risk Score",
        "department_name": "Department",
        "business_unit": "Business Unit"
    }
)

fig_department_risk.update_layout(
    height=520,
    margin=dict(l=40, r=40, t=70, b=40)
)

# ------------------------------------------------------------
# Figure 6: Monthly audit risk trend
# ------------------------------------------------------------

fig_monthly_trend = px.line(
    monthly_audit_trend,
    x="Month",
    y="Average_Risk_Score",
    markers=True,
    title="Monthly Average Audit Risk Score",
    labels={
        "Month": "Month",
        "Average_Risk_Score": "Average Audit Risk Score"
    }
)

fig_monthly_trend.update_layout(
    height=430,
    margin=dict(l=40, r=40, t=70, b=40)
)

# ------------------------------------------------------------
# Figure 7: ML anomaly distribution
# ------------------------------------------------------------

fig_ml_distribution = px.pie(
    anomaly_distribution,
    names="Label",
    values="Invoices",
    hole=0.45,
    title="Machine Learning Anomaly Detection Distribution"
)

fig_ml_distribution.update_layout(
    height=430,
    margin=dict(l=40, r=40, t=70, b=40)
)

# ------------------------------------------------------------
# Figure 8: Audit risk vs ML anomaly score
# ------------------------------------------------------------

fig_audit_ml_scatter = px.scatter(
    ml_df,
    x="audit_risk_score",
    y="ml_anomaly_score",
    color="ml_review_priority",
    size="invoice_amount",
    hover_name="invoice_id",
    hover_data=[
        "supplier_name",
        "department_name",
        "invoice_amount",
        "audit_risk_level",
        "ml_anomaly_flag",
        "primary_risk_driver"
    ],
    title="Audit Rules Risk Score vs ML Anomaly Score",
    labels={
        "audit_risk_score": "Audit Rules Risk Score",
        "ml_anomaly_score": "ML Anomaly Score",
        "ml_review_priority": "Review Priority"
    }
)

fig_audit_ml_scatter.update_layout(
    height=560,
    margin=dict(l=40, r=40, t=70, b=40)
)

# ------------------------------------------------------------
# Figure 9: Supplier anomalies
# ------------------------------------------------------------

fig_supplier_anomalies = px.bar(
    supplier_anomaly_summary.head(15).sort_values("Anomaly_Count", ascending=True),
    x="Anomaly_Count",
    y="supplier_name",
    orientation="h",
    color="supplier_category",
    title="Top 15 Suppliers by Detected ML Anomalies",
    labels={
        "Anomaly_Count": "Detected Anomalies",
        "supplier_name": "Supplier",
        "supplier_category": "Supplier Category"
    }
)

fig_supplier_anomalies.update_layout(
    height=560,
    margin=dict(l=40, r=40, t=70, b=40)
)

# ------------------------------------------------------------
# Figure 10: Department anomalies
# ------------------------------------------------------------

fig_department_anomalies = px.bar(
    department_anomaly_summary.sort_values("Anomaly_Count", ascending=True),
    x="Anomaly_Count",
    y="department_name",
    orientation="h",
    color="business_unit",
    title="Detected ML Anomalies by Department",
    labels={
        "Anomaly_Count": "Detected Anomalies",
        "department_name": "Department",
        "business_unit": "Business Unit"
    }
)

fig_department_anomalies.update_layout(
    height=520,
    margin=dict(l=40, r=40, t=70, b=40)
)

# ------------------------------------------------------------
# Prepare HTML tables
# ------------------------------------------------------------

priority_transactions_table = top_priority_transactions_excel.head(15).to_html(
    index=False,
    classes="data-table",
    border=0
)

supplier_risk_table = supplier_risk_summary[
    [
        "supplier_name",
        "supplier_category",
        "supplier_country",
        "Invoice_Count",
        "Total_Invoice_Amount",
        "Exception_Rate",
        "supplier_risk_score",
        "supplier_risk_level",
        "primary_supplier_risk_driver"
    ]
].head(15).to_html(
    index=False,
    classes="data-table",
    border=0
)

audit_rules_table = audit_rules_summary_df[
    [
        "Rule_Code",
        "Rule_Name",
        "Risk_Category",
        "Severity",
        "Flagged_Invoices",
        "Flagged_Percentage",
        "Total_Invoice_Amount"
    ]
].to_html(
    index=False,
    classes="data-table",
    border=0
)

# ------------------------------------------------------------
# Convert charts to HTML
# ------------------------------------------------------------

risk_distribution_html = plot_to_html(fig_risk_distribution)
audit_rules_html = plot_to_html(fig_audit_rules)
audit_categories_html = plot_to_html(fig_audit_categories)
supplier_risk_html = plot_to_html(fig_supplier_risk)
department_risk_html = plot_to_html(fig_department_risk)
monthly_trend_html = plot_to_html(fig_monthly_trend)
ml_distribution_html = plot_to_html(fig_ml_distribution)
audit_ml_scatter_html = plot_to_html(fig_audit_ml_scatter)
supplier_anomalies_html = plot_to_html(fig_supplier_anomalies)
department_anomalies_html = plot_to_html(fig_department_anomalies)

# ------------------------------------------------------------
# Build dashboard HTML
# ------------------------------------------------------------

dashboard_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Financial Audit Risk Analytics Platform</title>
    <script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>
    <style>
        :root {{
            --navy: #0f172a;
            --blue: #2563eb;
            --light-blue: #eff6ff;
            --green: #10b981;
            --orange: #f59e0b;
            --red: #ef4444;
            --gray: #64748b;
            --light-gray: #f8fafc;
            --border: #e2e8f0;
        }}

        * {{
            box-sizing: border-box;
        }}

        body {{
            margin: 0;
            font-family: Arial, Helvetica, sans-serif;
            background: linear-gradient(180deg, #f8fafc 0%, #eef2ff 100%);
            color: var(--navy);
        }}

        .hero {{
            padding: 46px 64px;
            background: linear-gradient(135deg, #0f172a 0%, #1d4ed8 100%);
            color: white;
        }}

        .hero h1 {{
            font-size: 42px;
            margin: 0 0 14px 0;
            letter-spacing: -0.5px;
        }}

        .hero p {{
            font-size: 18px;
            line-height: 1.6;
            max-width: 1100px;
            margin: 0;
        }}

        .hero-meta {{
            margin-top: 24px;
            font-size: 15px;
            color: #dbeafe;
        }}

        .container {{
            max-width: 1320px;
            margin: 0 auto;
            padding: 34px;
        }}

        .section {{
            background: white;
            border-radius: 22px;
            padding: 28px;
            margin-bottom: 28px;
            box-shadow: 0 12px 30px rgba(15, 23, 42, 0.08);
            border: 1px solid var(--border);
        }}

        .section h2 {{
            margin: 0 0 12px 0;
            font-size: 28px;
            border-left: 6px solid var(--blue);
            padding-left: 14px;
        }}

        .section p {{
            color: #334155;
            line-height: 1.7;
            font-size: 16px;
        }}

        .kpi-grid {{
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 18px;
            margin-top: 24px;
        }}

        .kpi-card {{
            background: linear-gradient(180deg, #ffffff 0%, #f8fafc 100%);
            border-radius: 18px;
            padding: 22px;
            border: 1px solid var(--border);
            box-shadow: 0 8px 18px rgba(15, 23, 42, 0.06);
        }}

        .kpi-label {{
            color: var(--gray);
            font-size: 14px;
            margin-bottom: 10px;
        }}

        .kpi-value {{
            font-size: 30px;
            font-weight: 800;
            color: var(--navy);
        }}

        .grid-2 {{
            display: grid;
            grid-template-columns: repeat(2, 1fr);
            gap: 24px;
            align-items: start;
        }}

        .chart-card {{
            background: #ffffff;
            border: 1px solid var(--border);
            border-radius: 18px;
            padding: 14px;
            overflow: hidden;
        }}

        .insight {{
            padding: 18px 22px;
            background: #ecfdf5;
            border-left: 6px solid var(--green);
            border-radius: 14px;
            margin-top: 22px;
            color: #064e3b;
            line-height: 1.6;
        }}

        .warning {{
            padding: 18px 22px;
            background: #fff7ed;
            border-left: 6px solid var(--orange);
            border-radius: 14px;
            margin-top: 22px;
            color: #7c2d12;
            line-height: 1.6;
        }}

        .data-table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
            margin-top: 18px;
        }}

        .data-table th {{
            background: #1e3a8a;
            color: white;
            padding: 10px;
            text-align: left;
            position: sticky;
            top: 0;
        }}

        .data-table td {{
            padding: 9px 10px;
            border-bottom: 1px solid var(--border);
            vertical-align: top;
        }}

        .data-table tr:nth-child(even) {{
            background: #f8fafc;
        }}

        .table-wrapper {{
            overflow-x: auto;
            border: 1px solid var(--border);
            border-radius: 14px;
            margin-top: 18px;
        }}

        footer {{
            text-align: center;
            color: var(--gray);
            padding: 28px;
            font-size: 14px;
        }}

        @media (max-width: 1000px) {{
            .kpi-grid {{
                grid-template-columns: repeat(2, 1fr);
            }}

            .grid-2 {{
                grid-template-columns: 1fr;
            }}

            .hero {{
                padding: 34px;
            }}

            .hero h1 {{
                font-size: 32px;
            }}
        }}

        @media (max-width: 600px) {{
            .kpi-grid {{
                grid-template-columns: 1fr;
            }}

            .container {{
                padding: 18px;
            }}
        }}
    </style>
</head>

<body>

    <div class="hero">
        <h1>Financial Audit Risk Analytics Platform</h1>
        <p>
            End-to-end audit analytics project combining financial data quality checks,
            audit rules engine, supplier risk scoring, SQL data mart, automated Excel reporting
            and machine learning anomaly detection.
        </p>
        <div class="hero-meta">
            <strong>Author:</strong> Chinez Benidir |
            <strong>Project Type:</strong> Audit Analytics, Risk Advisory & Financial Data Analytics
        </div>
    </div>

    <div class="container">

        <div class="section">
            <h2>Executive Audit KPI Overview</h2>
            <p>
                This section summarizes the financial exposure, audit exception volume,
                supplier risk and machine learning anomaly detection results.
            </p>

            <div class="kpi-grid">
                <div class="kpi-card">
                    <div class="kpi-label">Total Invoices</div>
                    <div class="kpi-value">{format_number(total_invoices_kpi)}</div>
                </div>
                <div class="kpi-card">
                    <div class="kpi-label">Total Invoice Amount</div>
                    <div class="kpi-value">{format_amount(total_amount_kpi)}</div>
                </div>
                <div class="kpi-card">
                    <div class="kpi-label">Audit Exception Rate</div>
                    <div class="kpi-value">{exception_rate_kpi}%</div>
                </div>
                <div class="kpi-card">
                    <div class="kpi-label">High/Critical Risk Invoices</div>
                    <div class="kpi-value">{format_number(high_risk_invoices_kpi)}</div>
                </div>
                <div class="kpi-card">
                    <div class="kpi-label">High/Critical Risk Exposure</div>
                    <div class="kpi-value">{format_amount(high_risk_exposure_kpi)}</div>
                </div>
                <div class="kpi-card">
                    <div class="kpi-label">ML Detected Anomalies</div>
                    <div class="kpi-value">{format_number(ml_anomalies_kpi)}</div>
                </div>
                <div class="kpi-card">
                    <div class="kpi-label">High-Risk Suppliers</div>
                    <div class="kpi-value">{format_number(high_risk_suppliers_kpi)}</div>
                </div>
                <div class="kpi-card">
                    <div class="kpi-label">Top Risk Supplier</div>
                    <div class="kpi-value" style="font-size:24px;">{top_supplier_kpi}</div>
                </div>
            </div>

            <div class="insight">
                <strong>Executive insight:</strong>
                The platform identifies a limited set of high-priority invoices and suppliers,
                allowing audit teams to focus their testing effort on the most relevant financial and control risks.
            </div>
        </div>

        <div class="section">
            <h2>Audit Risk Distribution</h2>
            <p>
                The audit risk score is based on structured audit rules covering procurement,
                approval controls, supplier master data, payment timing and duplicate payment risk.
            </p>
            <div class="grid-2">
                <div class="chart-card">{risk_distribution_html}</div>
                <div class="chart-card">{audit_categories_html}</div>
            </div>
        </div>

        <div class="section">
            <h2>Audit Rules Engine</h2>
            <p>
                The rules engine converts audit red flags into structured risk indicators,
                allowing transaction-level scoring and exception prioritization.
            </p>
            <div class="chart-card">{audit_rules_html}</div>

            <div class="table-wrapper">
                {audit_rules_table}
            </div>
        </div>

        <div class="section">
            <h2>Supplier Risk Scoring</h2>
            <p>
                Supplier-level scoring aggregates transaction exceptions, risk scores,
                critical rule triggers, spend concentration and supplier master data issues.
            </p>
            <div class="grid-2">
                <div class="chart-card">{supplier_risk_html}</div>
                <div class="chart-card">{supplier_anomalies_html}</div>
            </div>

            <div class="table-wrapper">
                {supplier_risk_table}
            </div>

            <div class="insight">
                <strong>Supplier insight:</strong>
                Supplier risk ranking helps prioritize third-party reviews and focus audit testing
                on suppliers with both exception concentration and financial exposure.
            </div>
        </div>

        <div class="section">
            <h2>Department and Business Unit Risk</h2>
            <p>
                Department-level risk analysis helps identify where internal controls,
                procurement workflows or approval processes may require additional review.
            </p>
            <div class="grid-2">
                <div class="chart-card">{department_risk_html}</div>
                <div class="chart-card">{department_anomalies_html}</div>
            </div>
        </div>

        <div class="section">
            <h2>Monthly Audit Risk Trend</h2>
            <p>
                The monthly trend tracks the evolution of the average audit risk score over the year.
            </p>
            <div class="chart-card">{monthly_trend_html}</div>
        </div>

        <div class="section">
            <h2>Machine Learning Anomaly Detection</h2>
            <p>
                Isolation Forest is used to identify unusual transactions that may not be fully captured
                by predefined audit rules.
            </p>
            <div class="grid-2">
                <div class="chart-card">{ml_distribution_html}</div>
                <div class="chart-card">{audit_ml_scatter_html}</div>
            </div>

            <div class="warning">
                <strong>Methodological note:</strong>
                Machine learning anomaly detection does not prove fraud. It highlights unusual transactions
                that should be reviewed together with audit rules, documentation and business context.
            </div>
        </div>

        <div class="section">
            <h2>High-Priority Transactions</h2>
            <p>
                These transactions are prioritized using a combined score that blends audit rules risk score
                and machine learning anomaly score.
            </p>

            <div class="table-wrapper">
                {priority_transactions_table}
            </div>
        </div>

        <div class="section">
            <h2>Business Conclusion</h2>
            <p>
                This project demonstrates an end-to-end audit analytics workflow:
                financial data preparation, audit rules design, risk scoring, supplier prioritization,
                anomaly detection, SQL data mart creation, Excel reporting and dashboarding.
            </p>

            <div class="insight">
                <strong>Recommended action:</strong>
                Prioritize detailed review of high-risk invoices, suppliers with high exception rates,
                transactions detected by machine learning anomalies, and departments showing elevated
                average risk scores.
            </div>
        </div>

    </div>

    <footer>
        Financial Audit Risk Analytics Platform | Chinez Benidir | Portfolio Project
    </footer>

</body>
</html>
"""

# ------------------------------------------------------------
# Save dashboard
# ------------------------------------------------------------

dashboard_path.write_text(dashboard_html, encoding="utf-8")

print("Interactive HTML dashboard created successfully")
print("Dashboard path:", dashboard_path)

INTERACTIVE HTML AUDIT DASHBOARD
Interactive HTML dashboard created successfully
Dashboard path: ..\dashboard\index.html
